#### Chile Groundwater Depth Prediction Analysis
#### Comprehensive ML pipeline for predicting depth to water using environmental and geographic features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time
import os

# Start timer
start_time = time.time()

# --- Configuration ---
# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Define the output folder for figures
output_folder = r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\python_code\chile_groundwater_results_database_analysis_7_8_25"
# Create the folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)
print(f"✅ Figures will be saved to: {output_folder}")

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("\n🌊 Chile Groundwater Database Analysis")
print("=" * 50)

# --- 1. DATA LOADING AND INITIAL EXPLORATION ---
print("\n📊 Loading and exploring the dataset...")

# Load the dataset
file_path = r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\results\chile_2025\groundwater_chile_and_elevation_dataset_2025_with_GEE_7_2_25.xlsx"
target_col = 'Depth to water (m)'

try:
    df = pd.read_csv(file_path) if file_path.endswith('.csv') else pd.read_excel(file_path)
    print(f"✅ Dataset loaded successfully! Shape: {df.shape}")
except Exception as e:
    print(f"❌ Error loading file: {e}")
    exit() # Exit if file cannot be loaded

# Filter data to only include rows where Status is blank/NaN
print(f"\nOriginal dataset shape: {df.shape}")
print(f"Rows with blank Status: {df['Status'].isnull().sum()}")
df_filtered = df[df['Status'].isnull()].copy()
print(f"Filtered dataset shape (Status is blank): {df_filtered.shape}")

if len(df_filtered) == 0:
    print("⚠️ No rows with blank Status found! Using the entire dataset for analysis.")
    df_filtered = df.copy()

# --- 2. DATA PREPROCESSING ---
print("\n🔧 Preprocessing data for analysis...")

# --- NEW: Filter for depth to water >= 0 ---
initial_rows = len(df_filtered)
df_filtered = df_filtered[df_filtered[target_col] >= 0].copy()
rows_removed = initial_rows - len(df_filtered)
if rows_removed > 0:
    print(f"Removed {rows_removed} rows with negative 'Depth to water (m)' values.")
print(f"Shape after filtering for non-negative depths: {df_filtered.shape}")

# Remove rows where target variable is missing
initial_rows = len(df_filtered)
df_filtered.dropna(subset=[target_col], inplace=True)
rows_removed = initial_rows - len(df_filtered)
if rows_removed > 0:
    print(f"Removed {rows_removed} rows with missing target values.")

# Identify numeric and categorical columns
numeric_cols = df_filtered.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df_filtered.select_dtypes(include=['object', 'category']).columns.tolist()

# Clean up column lists
if target_col in numeric_cols:
    numeric_cols.remove(target_col)
if 'Status' in categorical_cols:
    categorical_cols.remove('Status')

print(f"\nIdentified {len(numeric_cols)} numeric features and {len(categorical_cols)} categorical features.")

# Handle missing values in feature columns
for col in numeric_cols:
    if df_filtered[col].isnull().sum() > 0:
        df_filtered[col].fillna(df_filtered[col].median(), inplace=True)
for col in categorical_cols:
    if df_filtered[col].isnull().sum() > 0:
        df_filtered[col].fillna(df_filtered[col].mode()[0], inplace=True)

print("✅ Preprocessing complete.")

# --- 3. EXPLORATORY DATA ANALYSIS (EDA) & VISUALIZATION ---
print("\n🎨 Creating and saving visualizations...")

# Plot 1: Target Variable Distribution
plt.figure(figsize=(14, 6))
plt.suptitle('Distribution of Depth to Water (m >= 0)', fontsize=16)
plt.subplot(1, 2, 1)
sns.histplot(df_filtered[target_col], bins=50, kde=True, color='skyblue')
plt.title('Original Distribution')
plt.xlabel('Depth to Water (m)')
plt.ylabel('Frequency')
plt.subplot(1, 2, 2)
log_target = np.log1p(df_filtered[target_col])
sns.histplot(log_target, bins=50, kde=True, color='lightcoral')
plt.title('Log-Transformed Distribution')
plt.xlabel('Log(1 + Depth to Water)')
plt.ylabel('Frequency')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
fig_path = os.path.join(output_folder, '1_target_variable_distribution.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")

# Plot 2: Correlation Matrix of Numeric Features
plt.figure(figsize=(16, 14))
corr_df = df_filtered[numeric_cols + [target_col]].corr()
mask = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(corr_df, mask=mask, annot=True, cmap='coolwarm', fmt='.2f', linewidths=.5)
plt.title('Correlation Matrix of Numeric Features', fontsize=16)
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
fig_path = os.path.join(output_folder, '2_correlation_heatmap.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")

# Plot 3: Scatter Plots of Top Correlated Features vs. Target
correlations = df_filtered[numeric_cols].corrwith(df_filtered[target_col]).abs().sort_values(ascending=False)
top_features = correlations.head(8).index
plt.figure(figsize=(20, 10))
plt.suptitle('Depth to Water vs. Top Correlated Numeric Features', fontsize=16)
for i, feature in enumerate(top_features):
    plt.subplot(2, 4, i + 1)
    sns.scatterplot(x=df_filtered[feature], y=df_filtered[target_col], alpha=0.6, s=15, edgecolor=None)
    plt.title(f'vs. {feature}')
    plt.xlabel(feature)
    plt.ylabel('Depth to Water (m)' if i % 4 == 0 else '')
    corr_coef = df_filtered[feature].corr(df_filtered[target_col])
    plt.text(0.05, 0.95, f'r = {corr_coef:.3f}', transform=plt.gca().transAxes,
             verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
fig_path = os.path.join(output_folder, '3_top_features_scatter_plots.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")

# --- MODIFIED: Plot 4: Analysis of Categorical Features (Basin) ---
if 'Basin' in categorical_cols:
    print("\nGenerating improved categorical analysis plot for 'Basin'...")
    cat_to_plot = 'Basin'
    
    # Get the top 10 basins by count
    top_categories = df_filtered[cat_to_plot].value_counts().nlargest(10).index
    plot_df = df_filtered[df_filtered[cat_to_plot].isin(top_categories)]

    # Get counts for labeling
    counts = plot_df[cat_to_plot].value_counts()
    
    # Create new labels with counts
    # Ensure the labels are in the same order as the plot
    new_labels = [f"{cat}\n(n={counts[cat]})" for cat in top_categories]

    # Create the plot
    fig, ax = plt.subplots(figsize=(18, 10))
    sns.boxplot(data=plot_df, x=cat_to_plot, y=target_col, order=top_categories,
                showfliers=False, ax=ax) # Hiding outliers for clarity

    ax.set_title(f'Depth to Water by Basin (Top 10 by Record Count)', fontsize=18, pad=20)
    ax.set_xlabel(cat_to_plot, fontsize=12)
    ax.set_ylabel('Depth to Water (m)', fontsize=12)
    ax.set_xticklabels(new_labels, rotation=45, ha='right')

    # --- NEW: Add explanation text to the figure ---
    explanation_text = (
        "How to Read This Plot:\n"
        "■ The Box: Shows the middle 50% of data (25th to 75th percentile).\n"
        "■ The Line in the Box: Represents the median depth.\n"
        "■ The Whiskers: Show the data range (excluding outliers).\n"
        "■ (n=...): The number of data records for that basin."
    )
    fig.text(0.99, 0.97, explanation_text, transform=ax.transAxes, fontsize=10,
             verticalalignment='top', horizontalalignment='right',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    plt.tight_layout()
    fig_path = os.path.join(output_folder, '4_categorical_analysis_Basin_Top10.png')
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fig_path}")
else:
    print("\n'Basin' column not found. Skipping categorical analysis plot.")

# --- 4. SUMMARY OF DATABASE ANALYSIS ---
print("\n📋 SUMMARY OF DATABASE ANALYSIS")
print("=" * 50)
print(f"✅ Analysis performed on {df_filtered.shape[0]} data points with {len(numeric_cols) + len(categorical_cols)} features.")
print(f"🎯 Target Variable '{target_col}':")
print(f"   - Mean: {df_filtered[target_col].mean():.2f} m")
print(f"   - Median: {df_filtered[target_col].median():.2f} m")
print(f"   - Standard Deviation: {df_filtered[target_col].std():.2f} m")
print(f"   - Range: {df_filtered[target_col].min():.2f} m to {df_filtered[target_col].max():.2f} m")

print("\n🔑 Key Findings from EDA:")
if not correlations.empty:
    print("   - Top 5 most correlated numeric features with Depth to Water:")
    for i, (feature, corr_val) in enumerate(correlations.head(5).items()):
        print(f"     {i+1}. {feature} (Correlation: {corr_val:.3f})")
else:
    print("   - No numeric correlations to report.")

if 'Basin' in categorical_cols:
     print(f"   - The 'Basin' feature shows notable variation in median water depth across different river basins.")

print("\n💡 Recommendations based on analysis:")
print("   1. The distribution of 'Depth to water (m)' is highly right-skewed. Log-transformation may be beneficial for statistical modeling.")
if not correlations.empty and len(correlations) > 1:
    print(f"   2. Features like '{correlations.index[0]}' and '{correlations.index[1]}' show the strongest linear relationships and are likely key drivers.")
print("   3. Investigate the basins with high variability in water depth (larger boxes) to understand local hydrogeological factors.")
print("   4. The number of records per basin varies significantly; conclusions for basins with low 'n' should be made with caution.")

# --- End of Script ---
end_time = time.time()
execution_time = (end_time - start_time)
print(f"\n⏱️ Total execution time: {execution_time:.2f} seconds")
print(f"\n🎉 Analysis completed successfully!")

### General maps

In [ ]:
#!/usr/bin/env python3
"""
Professional Groundwater Well Visualization for Scientific Publication

This module creates publication-quality maps of groundwater well data in Chile,
designed for scientific journals (e.g., ScienceDirect). The code follows
best practices for reproducible research and generates high-quality figures
suitable for peer-reviewed publications.

Author: [Your Name]
Institution: [Your Institution]
Date: July 2025
License: MIT

Dependencies:
    - pandas >= 1.3.0
    - geopandas >= 0.10.0
    - matplotlib >= 3.5.0
    - contextily >= 1.2.0
    - shapely >= 1.8.0
    - numpy >= 1.21.0
"""

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm
import contextily as cx
import numpy as np
import os
import warnings
import logging
from pathlib import Path
from typing import Tuple, Optional, List, Dict, Any
import seaborn as sns

# Configure scientific plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Suppress non-critical warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

class GroundwaterVisualizationError(Exception):
    """Custom exception for groundwater visualization errors."""
    pass

class GroundwaterMapper:
    """
    A class for creating professional groundwater well visualizations.
    
    This class provides methods to create publication-quality maps of groundwater
    well data with proper cartographic elements and scientific styling.
    """
    
    def __init__(self, data_path: str, output_dir: str):
        """
        Initialize the GroundwaterMapper.
        
        Args:
            data_path (str): Path to the Excel file containing groundwater data
            output_dir (str): Directory where output maps will be saved
        """
        self.data_path = Path(data_path)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        
        # Column name mappings for clarity
        self.column_mapping = {
            'longitude': 'Longitude_GCS_WGS_1984',
            'latitude': 'Latitude_GCS_WGS_1984',
            'depth': 'Depth to water (m)',
            'status': 'Status',
            'basin': 'Basin'
        }
        
        # Scientific publication parameters
        self.figure_params = {
            'dpi': 300,
            'bbox_inches': 'tight',
            'facecolor': 'white',
            'edgecolor': 'none'
        }
        
        self.style_params = {
            'title_fontsize': 14,
            'label_fontsize': 12,
            'legend_fontsize': 10,
            'tick_fontsize': 10
        }
        
        logger.info(f"GroundwaterMapper initialized. Output directory: {self.output_dir}")
    
    def load_and_preprocess_data(self) -> gpd.GeoDataFrame:
        """
        Load and preprocess groundwater data with scientific quality control.
        
        Returns:
            gpd.GeoDataFrame: Cleaned and validated groundwater data
            
        Raises:
            GroundwaterVisualizationError: If data loading or preprocessing fails
        """
        try:
            # Load data
            df = pd.read_excel(self.data_path)
            logger.info(f"Data loaded successfully. Initial shape: {df.shape}")
            
            # Quality control filters
            initial_count = len(df)
            
            # Filter 1: Remove records with non-null status (likely inactive wells)
            df = df[df[self.column_mapping['status']].isnull()].copy()
            logger.info(f"Removed {initial_count - len(df)} records with non-null status")
            
            # Filter 2: Remove invalid depth measurements
            df = df[df[self.column_mapping['depth']] >= 0].copy()
            logger.info(f"Removed records with negative depth values")
            
            # Filter 3: Remove records with missing coordinates
            df = df.dropna(subset=[self.column_mapping['longitude'], 
                                 self.column_mapping['latitude']])
            logger.info(f"Final dataset shape after quality control: {df.shape}")
            
            # Validate coordinate ranges for Chile
            lon_range = (-110, -66)  # Chile longitude range
            lat_range = (-56, -17)   # Chile latitude range
            
            coord_mask = (
                (df[self.column_mapping['longitude']] >= lon_range[0]) &
                (df[self.column_mapping['longitude']] <= lon_range[1]) &
                (df[self.column_mapping['latitude']] >= lat_range[0]) &
                (df[self.column_mapping['latitude']] <= lat_range[1])
            )
            
            df = df[coord_mask].copy()
            logger.info(f"Coordinates validated. Final shape: {df.shape}")
            
            # Create GeoDataFrame
            gdf = gpd.GeoDataFrame(
                df,
                geometry=gpd.points_from_xy(
                    df[self.column_mapping['longitude']],
                    df[self.column_mapping['latitude']]
                ),
                crs="EPSG:4326"
            )
            
            # Generate descriptive statistics
            self._generate_data_summary(gdf)
            
            return gdf
            
        except Exception as e:
            raise GroundwaterVisualizationError(f"Data preprocessing failed: {str(e)}")
    
    def _generate_data_summary(self, gdf: gpd.GeoDataFrame) -> None:
        """Generate and log descriptive statistics for the dataset."""
        depth_stats = gdf[self.column_mapping['depth']].describe()
        logger.info("Dataset Summary:")
        logger.info(f"  - Total wells: {len(gdf)}")
        logger.info(f"  - Depth statistics (m):")
        logger.info(f"    Mean: {depth_stats['mean']:.2f}")
        logger.info(f"    Median: {depth_stats['50%']:.2f}")
        logger.info(f"    Std: {depth_stats['std']:.2f}")
        logger.info(f"    Range: {depth_stats['min']:.2f} - {depth_stats['max']:.2f}")
        
        if self.column_mapping['basin'] in gdf.columns:
            basin_count = gdf[self.column_mapping['basin']].nunique()
            logger.info(f"  - Number of basins: {basin_count}")
    
    def _add_cartographic_elements(self, ax: plt.Axes, crs: str, 
                                  scale_length: int = 200) -> None:
        """
        Add professional cartographic elements to the map.
        
        Args:
            ax: Matplotlib axes object
            crs: Coordinate reference system
            scale_length: Length of scale bar in kilometers
        """
        # North arrow
        ax.annotate('N', xy=(0.95, 0.95), xytext=(0.95, 0.90),
                   arrowprops=dict(facecolor='black', width=2, headwidth=8),
                   ha='center', va='center', fontsize=self.style_params['label_fontsize'],
                   fontweight='bold', xycoords='axes fraction')
        
        # Scale bar
        fontprops = fm.FontProperties(size=self.style_params['legend_fontsize'])
        scalebar = AnchoredSizeBar(
            ax.transData,
            scale_length * 1000,  # Convert to meters
            f'{scale_length} km',
            'lower right',
            pad=0.3,
            color='black',
            frameon=False,
            size_vertical=3,
            fontproperties=fontprops
        )
        ax.add_artist(scalebar)
        
        # Grid
        ax.grid(True, linestyle='--', alpha=0.3, linewidth=0.5)
        ax.set_axisbelow(True)
    
    def _add_basemap_with_fallback(self, ax: plt.Axes, crs: str) -> bool:
        """
        Add basemap with multiple provider fallbacks.
        
        Args:
            ax: Matplotlib axes object
            crs: Coordinate reference system
            
        Returns:
            bool: True if basemap was successfully added
        """
        providers = [
            cx.providers.CartoDB.Positron,
            cx.providers.OpenStreetMap.Mapnik,
            cx.providers.CartoDB.Voyager,
            cx.providers.Esri.WorldGrayCanvas
        ]
        
        for i, provider in enumerate(providers):
            try:
                cx.add_basemap(ax, crs=crs, source=provider, alpha=0.7)
                logger.info(f"Basemap loaded successfully (provider {i+1})")
                return True
            except Exception as e:
                logger.warning(f"Basemap provider {i+1} failed: {str(e)}")
                continue
        
        logger.warning("All basemap providers failed. Creating map without basemap.")
        return False
    
    def create_well_location_map(self, gdf: gpd.GeoDataFrame) -> str:
        """
        Create a publication-quality map showing general well locations.
        
        Args:
            gdf: GeoDataFrame containing well data
            
        Returns:
            str: Path to the saved figure
        """
        logger.info("Creating well location map...")
        
        # Use unique locations to avoid overplotting
        gdf_unique = gdf.drop_duplicates(subset=[self.column_mapping['longitude'],
                                               self.column_mapping['latitude']])
        
        # Create figure with scientific proportions
        fig, ax = plt.subplots(1, 1, figsize=(8, 12))
        
        # Project to Web Mercator for basemap compatibility
        gdf_mercator = gdf_unique.to_crs(epsg=3857)
        
        # Plot wells
        gdf_mercator.plot(
            ax=ax,
            marker='o',
            color='#d62728',  # Scientific red
            markersize=15,
            alpha=0.7,
            edgecolor='black',
            linewidth=0.3,
            label='Groundwater Wells'
        )
        
        # Add basemap
        self._add_basemap_with_fallback(ax, gdf_mercator.crs)
        
        # Add cartographic elements
        self._add_cartographic_elements(ax, gdf_mercator.crs)
        
        # Styling
        ax.set_title('Distribution of Groundwater Wells in Chile', 
                    fontsize=self.style_params['title_fontsize'], 
                    fontweight='bold', pad=20)
        ax.set_xlabel('Longitude', fontsize=self.style_params['label_fontsize'])
        ax.set_ylabel('Latitude', fontsize=self.style_params['label_fontsize'])
        
        # Add legend
        legend_elements = [mpatches.Patch(color='#d62728', label=f'Wells (n={len(gdf_unique)})')]
        ax.legend(handles=legend_elements, 
                 loc='upper right', 
                 fontsize=self.style_params['legend_fontsize'],
                 frameon=True,
                 fancybox=True,
                 shadow=True)
        
        # Adjust layout
        plt.tight_layout()
        
        # Save figure
        output_path = self.output_dir / 'figure_1_well_locations.png'
        plt.savefig(output_path, **self.figure_params)
        plt.savefig(output_path.with_suffix('.pdf'), **self.figure_params)  # Also save as PDF
        
        plt.show()
        logger.info(f"Well location map saved to: {output_path}")
        return str(output_path)
    
    def create_basin_analysis_map(self, gdf: gpd.GeoDataFrame) -> Optional[str]:
        """
        Create a publication-quality map showing basin-level analysis.
        
        Args:
            gdf: GeoDataFrame containing well data
            
        Returns:
            Optional[str]: Path to the saved figure, or None if basin data unavailable
        """
        if self.column_mapping['basin'] not in gdf.columns:
            logger.warning("Basin column not found. Skipping basin analysis map.")
            return None
        
        logger.info("Creating basin analysis map...")
        
        # Aggregate by basin
        basin_stats = gdf.groupby(self.column_mapping['basin']).agg({
            self.column_mapping['depth']: ['mean', 'count', 'std'],
            self.column_mapping['longitude']: 'mean',
            self.column_mapping['latitude']: 'mean'
        }).reset_index()
        
        # Flatten column names
        basin_stats.columns = ['basin', 'depth_mean', 'well_count', 'depth_std', 
                              'longitude', 'latitude']
        
        # Create GeoDataFrame
        gdf_basins = gpd.GeoDataFrame(
            basin_stats,
            geometry=gpd.points_from_xy(basin_stats.longitude, basin_stats.latitude),
            crs="EPSG:4326"
        )
        
        # Create figure
        fig, ax = plt.subplots(1, 1, figsize=(8, 12))
        
        # Project to Web Mercator
        gdf_basins_mercator = gdf_basins.to_crs(epsg=3857)
        
        # Create size scaling for point sizes
        size_scale = 100  # Base size
        sizes = gdf_basins['well_count'] * size_scale / gdf_basins['well_count'].max() + 20
        
        # Plot with scientific colormap
        scatter = ax.scatter(
            gdf_basins_mercator.geometry.x,
            gdf_basins_mercator.geometry.y,
            c=gdf_basins['depth_mean'],
            s=sizes,
            cmap='viridis_r',  # Scientific colormap
            alpha=0.8,
            edgecolor='black',
            linewidth=0.5
        )
        
        # Add basemap
        self._add_basemap_with_fallback(ax, gdf_basins_mercator.crs)
        
        # Add cartographic elements
        self._add_cartographic_elements(ax, gdf_basins_mercator.crs)
        
        # Color bar
        cbar = plt.colorbar(scatter, ax=ax, orientation='horizontal', 
                           pad=0.02, aspect=50, shrink=0.8)
        cbar.set_label('Mean Depth to Water (m)', 
                      fontsize=self.style_params['label_fontsize'])
        cbar.ax.tick_params(labelsize=self.style_params['tick_fontsize'])
        
        # Size legend
        legend_sizes = [10, 100, 500]
        legend_handles = []
        for size in legend_sizes:
            if size <= gdf_basins['well_count'].max():
                scaled_size = size * size_scale / gdf_basins['well_count'].max() + 20
                legend_handles.append(
                    plt.scatter([], [], s=scaled_size, c='gray', alpha=0.8,
                               edgecolor='black', linewidth=0.5,
                               label=f'{size} wells')
                )
        
        if legend_handles:
            ax.legend(handles=legend_handles, 
                     title='Well Count', 
                     loc='upper right',
                     fontsize=self.style_params['legend_fontsize'],
                     title_fontsize=self.style_params['legend_fontsize'],
                     frameon=True,
                     fancybox=True,
                     shadow=True)
        
        # Styling
        ax.set_title('Basin-Scale Groundwater Depth Analysis', 
                    fontsize=self.style_params['title_fontsize'], 
                    fontweight='bold', pad=20)
        ax.set_xlabel('Longitude', fontsize=self.style_params['label_fontsize'])
        ax.set_ylabel('Latitude', fontsize=self.style_params['label_fontsize'])
        
        # Adjust layout
        plt.tight_layout()
        
        # Save figure
        output_path = self.output_dir / 'figure_2_basin_analysis.png'
        plt.savefig(output_path, **self.figure_params)
        plt.savefig(output_path.with_suffix('.pdf'), **self.figure_params)
        
        plt.show()
        logger.info(f"Basin analysis map saved to: {output_path}")
        return str(output_path)
    
    def generate_summary_statistics(self, gdf: gpd.GeoDataFrame) -> str:
        """
        Generate summary statistics table for the manuscript.
        
        Args:
            gdf: GeoDataFrame containing well data
            
        Returns:
            str: Path to the saved statistics file
        """
        logger.info("Generating summary statistics...")
        
        # Overall statistics
        overall_stats = {
            'Parameter': ['Total Wells', 'Mean Depth (m)', 'Median Depth (m)', 
                         'Standard Deviation (m)', 'Minimum Depth (m)', 'Maximum Depth (m)'],
            'Value': [
                len(gdf),
                f"{gdf[self.column_mapping['depth']].mean():.2f}",
                f"{gdf[self.column_mapping['depth']].median():.2f}",
                f"{gdf[self.column_mapping['depth']].std():.2f}",
                f"{gdf[self.column_mapping['depth']].min():.2f}",
                f"{gdf[self.column_mapping['depth']].max():.2f}"
            ]
        }
        
        stats_df = pd.DataFrame(overall_stats)
        
        # Basin statistics if available
        if self.column_mapping['basin'] in gdf.columns:
            basin_stats = gdf.groupby(self.column_mapping['basin']).agg({
                self.column_mapping['depth']: ['count', 'mean', 'std', 'min', 'max']
            }).round(2)
            
            basin_stats.columns = ['Well_Count', 'Mean_Depth_m', 'Std_Depth_m', 
                                  'Min_Depth_m', 'Max_Depth_m']
            basin_stats = basin_stats.reset_index()
            
            # Save basin statistics
            basin_output_path = self.output_dir / 'basin_statistics.csv'
            basin_stats.to_csv(basin_output_path, index=False)
            logger.info(f"Basin statistics saved to: {basin_output_path}")
        
        # Save overall statistics
        output_path = self.output_dir / 'summary_statistics.csv'
        stats_df.to_csv(output_path, index=False)
        logger.info(f"Summary statistics saved to: {output_path}")
        
        return str(output_path)


def main():
    """
    Main function to execute the groundwater visualization workflow.
    """
    # Configuration
    data_file_path = r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\results\chile_2025\groundwater_chile_and_elevation_dataset_2025_with_GEE_7_2_25.xlsx"
    output_folder = r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\python_code\chile_groundwater_results_mapping_professional_7_8_25"
    
    try:
        # Initialize mapper
        mapper = GroundwaterMapper(data_file_path, output_folder)
        
        # Load and preprocess data
        gdf = mapper.load_and_preprocess_data()
        
        # Create visualizations
        well_map_path = mapper.create_well_location_map(gdf)
        basin_map_path = mapper.create_basin_analysis_map(gdf)
        
        # Generate statistics
        stats_path = mapper.generate_summary_statistics(gdf)
        
        # Summary
        logger.info("Analysis completed successfully!")
        logger.info(f"Well location map: {well_map_path}")
        if basin_map_path:
            logger.info(f"Basin analysis map: {basin_map_path}")
        logger.info(f"Summary statistics: {stats_path}")
        
    except GroundwaterVisualizationError as e:
        logger.error(f"Visualization error: {e}")
    except Exception as e:
        logger.error(f"Unexpected error: {e}")


if __name__ == "__main__":
    main()

### ML process

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance
import warnings
import time
import os
from joblib import Parallel, delayed

# --- SETUP ---
warnings.filterwarnings('ignore')
start_time = time.time()

# Create a directory to save results
OUTPUT_DIR = r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\python_code\chile_groundwater_results_7_8_25"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("🌊 Chile Groundwater Depth Prediction Analysis")
print("=" * 50)

# --- HELPER FUNCTIONS ---
def mean_absolute_percentage_error(y_true, y_pred):
    """Calculate Mean Absolute Percentage Error (MAPE)"""
    epsilon = 1e-10
    return np.mean(np.abs((y_true - y_pred) / (y_true + epsilon))) * 100

def train_and_evaluate_model(name, model, X_train, y_train, X_test, y_test, scaler):
    """Function to train a single model, make predictions, and calculate metrics."""
    print(f"  Training {name}...")
    
    # Use scaled data for specific models
    if name in ['Linear Regression', 'Ridge Regression', 'Lasso Regression', 
                'Elastic Net', 'Support Vector Regression', 'K-Nearest Neighbors']:
        X_train_model = scaler.transform(X_train)
        X_test_model = scaler.transform(X_test)
    else:
        X_train_model = X_train
        X_test_model = X_test

    # Train model
    model.fit(X_train_model, y_train)
    
    # Make predictions
    y_pred_train = model.predict(X_train_model)
    y_pred_test = model.predict(X_test_model)
    
    # Calculate metrics
    results = {
        'Train_RMSE': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'Train_R²': r2_score(y_train, y_pred_train),
        'Train_MAPE': mean_absolute_percentage_error(y_train, y_pred_train),
        'Test_RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'Test_R²': r2_score(y_test, y_pred_test),
        'Test_MAPE': mean_absolute_percentage_error(y_test, y_pred_test),
    }
    
    predictions = {
        'train': y_pred_train,
        'test': y_pred_test
    }
    
    return name, results, predictions, model

# --- 1. DATA LOADING AND INITIAL EXPLORATION ---
print("\n📊 1. Loading and exploring the dataset...")

# Load the dataset
# IMPORTANT: Replace this with the actual path to your file
file_path = r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\results\chile_2025\groundwater_chile_and_elevation_dataset_2025_with_GEE_7_2_25.xlsx"

try:
    df = pd.read_csv(file_path) if file_path.endswith('.csv') else pd.read_excel(file_path)
    print(f"✅ Dataset loaded successfully! Initial shape: {df.shape}")
except Exception as e:
    print(f"❌ Error loading file: {e}")
    # Create a sample dataset for demonstration if loading fails
    print("Creating sample data for demonstration...")
    np.random.seed(42)
    n_samples = 1000
    df = pd.DataFrame({
        'Depth to water (m)': np.random.exponential(10, n_samples) - 5, # Add some negative values
        'Elevation': np.random.normal(500, 200, n_samples),
        'Longitude_GCS_WGS_1984': np.random.uniform(-75, -65, n_samples),
        'Latitude_GCS_WGS_1984': np.random.uniform(-35, -25, n_samples),
        'Status': np.random.choice(['Active', 'Inactive', np.nan], n_samples, p=[0.3, 0.3, 0.4])
    })

# --- 2. DATA PREPROCESSING ---
print("\n🔧 2. Data preprocessing...")

target_col = 'Depth to water (m)'

# Filter to only include rows where Status is blank/NaN
df_filtered = df[df['Status'].isnull()].copy()
print(f"Filtered to blank Status rows. Shape: {df_filtered.shape}")

# Filter to only use depth to water values equal or higher than 0
initial_rows = len(df_filtered)
df_filtered = df_filtered[df_filtered[target_col] >= 0].copy()
print(f"Removed {initial_rows - len(df_filtered)} rows with negative Depth to Water.")

# Print total records and unique wells
print(f"✅ Total records for analysis: {len(df_filtered)}")
df_filtered['well_id'] = df_filtered['Longitude_GCS_WGS_1984'].astype(str) + '_' + df_filtered['Latitude_GCS_WGS_1984'].astype(str)
unique_wells = df_filtered['well_id'].nunique()
print(f"✅ Total unique well locations: {unique_wells}")

# Remove rows where target variable is missing
df_filtered.dropna(subset=[target_col], inplace=True)

# Identify feature columns
numeric_cols = df_filtered.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_filtered.select_dtypes(include=['object']).columns.tolist()

# Clean up column lists
cols_to_remove = [target_col, 'Status', 'well_id']
numeric_cols = [col for col in numeric_cols if col not in cols_to_remove]
categorical_cols = [col for col in categorical_cols if col not in cols_to_remove]

# Handle missing values and encode categorical variables
for col in numeric_cols:
    df_filtered[col].fillna(df_filtered[col].median(), inplace=True)
for col in categorical_cols:
    df_filtered[col].fillna(df_filtered[col].mode()[0], inplace=True)
    df_filtered[col + '_encoded'] = LabelEncoder().fit_transform(df_filtered[col].astype(str))

# Create final feature matrix and target vector
encoded_categorical_cols = [col + '_encoded' for col in categorical_cols]
feature_cols = numeric_cols + encoded_categorical_cols
X = df_filtered[feature_cols].copy()
y = df_filtered[target_col].copy()

print(f"Final feature matrix shape: {X.shape}")

# --- 3. MODEL TRAINING AND EVALUATION ---
print("\n🤖 3. Training multiple ML models in parallel...")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Scale the features (fit on training data only)
scaler = StandardScaler().fit(X_train)

# Define models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'Elastic Net': ElasticNet(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'K-Nearest Neighbors': KNeighborsRegressor(),
    'Support Vector Regression': SVR()
}

# Use joblib to run model training in parallel
parallel_results = Parallel(n_jobs=-1)(
    delayed(train_and_evaluate_model)(name, model, X_train, y_train, X_test, y_test, scaler)
    for name, model in models.items()
)

# Unpack results from parallel execution
results = {res[0]: res[1] for res in parallel_results}
predictions = {res[0]: res[2] for res in parallel_results}
trained_models = {res[0]: res[3] for res in parallel_results}

print("✅ All models trained successfully.")

# --- 4. MODEL COMPARISON AND RESULTS ---
print("\n📊 4. Comparing model performance...")

# Create performance DataFrame
results_df = pd.DataFrame.from_dict(results, orient='index')
results_df = results_df.sort_values(by='Test_MAPE', ascending=True)

# Save Model Performance Comparison Table to an excel file
excel_path = os.path.join(OUTPUT_DIR, 'model_performance_comparison.xlsx')
results_df.to_excel(excel_path)
print(f"✅ Model performance table saved to: {excel_path}")

# Identify best models
best_model_name = results_df.index[0]
top_3_models = results_df.index[:3].tolist()
print(f"\n🏆 Best Model: {best_model_name} (Lowest Test MAPE: {results_df.loc[best_model_name, 'Test_MAPE']:.2f}%)")
print(f"   Top 3 Models: {', '.join(top_3_models)}")

# --- 5. VISUALIZATIONS ---
print("\n🎨 5. Generating and saving visualizations...")

# Create comparison plots for R2, RMSE, and MAPE
metrics_to_plot = {'R²': 'R² Score', 'RMSE': 'RMSE (m)', 'MAPE': 'MAPE (%)'}
for metric, label in metrics_to_plot.items():
    plt.figure(figsize=(12, 7))
    
    train_scores = results_df[f'Train_{metric}']
    test_scores = results_df[f'Test_{metric}']
    
    x_pos = np.arange(len(results_df))
    width = 0.35
    
    plt.bar(x_pos - width/2, train_scores, width, label=f'Train {metric}', color='lightblue')
    plt.bar(x_pos + width/2, test_scores, width, label=f'Test {metric}', color='lightcoral')
    
    plt.xlabel('Models')
    plt.ylabel(label)
    plt.title(f'Training vs Testing {metric} Comparison')
    plt.xticks(x_pos, results_df.index, rotation=45, ha='right')
    plt.legend()
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    
    fig_path = os.path.join(OUTPUT_DIR, f'comparison_{metric}.png')
    plt.savefig(fig_path)
    plt.close() # Close the plot to free memory
    print(f"✅ {metric} comparison plot saved to: {fig_path}")

# >> MODIFIED SECTION <<
# Do one-to-one plots for the top 3 models (training and testing)
for model_name in top_3_models:
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle(f'Predicted vs. Actual: {model_name}', fontsize=16)

    # --- Training Data Plot ---
    # Retrieve metrics
    train_r2 = results_df.loc[model_name, 'Train_R²']
    train_mape = results_df.loc[model_name, 'Train_MAPE']
    train_rmse = results_df.loc[model_name, 'Train_RMSE']
    
    # Plot data
    axes[0].scatter(y_train, predictions[model_name]['train'], alpha=0.6, color='blue', s=20, label='Data points')
    axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2, label='1:1 Line')
    axes[0].set_xlabel('Actual Depth to Water (m)')
    axes[0].set_ylabel('Predicted Depth to Water (m)')
    # Add sample count to title
    axes[0].set_title(f'Training Sample Performance (n = {len(y_train)})')
    axes[0].grid(True, linestyle='--', alpha=0.6)
    
    # Add metrics text box (including RMSE)
    metrics_text_train = (f'R² = {train_r2:.3f}\n'
                          f'MAPE = {train_mape:.2f}%\n'
                          f'RMSE = {train_rmse:.2f} m')
    axes[0].text(0.05, 0.95, metrics_text_train, 
                 transform=axes[0].transAxes, fontsize=12, verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    axes[0].legend()

    # --- Testing Data Plot ---
    # Retrieve metrics
    test_r2 = results_df.loc[model_name, 'Test_R²']
    test_mape = results_df.loc[model_name, 'Test_MAPE']
    test_rmse = results_df.loc[model_name, 'Test_RMSE']

    # Plot data
    axes[1].scatter(y_test, predictions[model_name]['test'], alpha=0.6, color='green', s=20, label='Data points')
    axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='1:1 Line')
    axes[1].set_xlabel('Actual Depth to Water (m)')
    axes[1].set_ylabel('Predicted Depth to Water (m)')
    # Add sample count to title
    axes[1].set_title(f'Testing Sample Performance (n = {len(y_test)})')
    axes[1].grid(True, linestyle='--', alpha=0.6)
    
    # Add metrics text box (including RMSE)
    metrics_text_test = (f'R² = {test_r2:.3f}\n'
                         f'MAPE = {test_mape:.2f}%\n'
                         f'RMSE = {test_rmse:.2f} m')
    axes[1].text(0.05, 0.95, metrics_text_test, 
                 transform=axes[1].transAxes, fontsize=12, verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    axes[1].legend()

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    fig_path = os.path.join(OUTPUT_DIR, f'pred_vs_actual_{model_name.replace(" ", "_")}.png')
    plt.savefig(fig_path)
    plt.close()
    print(f"✅ Predicted vs. Actual plot for {model_name} saved to: {fig_path}")

# --- 6. FEATURE IMPORTANCE ANALYSIS ---
print(f"\n🎯 6. Feature Importance Analysis for {best_model_name}...")

best_model = trained_models[best_model_name]

# Use permutation importance for all models for consistency, or built-in if available
if hasattr(best_model, 'feature_importances_'):
    importance_scores = best_model.feature_importances_
    importance_type = "Built-in"
else:
    print("  Calculating permutation importance...")
    if best_model_name in ['Linear Regression', 'Ridge Regression', 'Lasso Regression', 
                           'Elastic Net', 'Support Vector Regression', 'K-Nearest Neighbors']:
        X_test_importance = scaler.transform(X_test)
    else:
        X_test_importance = X_test
        
    perm_importance = permutation_importance(best_model, X_test_importance, y_test, 
                                           n_repeats=10, random_state=42, n_jobs=-1)
    importance_scores = perm_importance.importances_mean
    importance_type = "Permutation"

feature_importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importance_scores
}).sort_values('Importance', ascending=False).reset_index(drop=True)

# Plot the top 10 feature importance variables
top_10_features = feature_importance_df.head(10)

plt.figure(figsize=(10, 8))
sns.barplot(x='Importance', y='Feature', data=top_10_features, palette='viridis')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.title(f'Top 10 Feature Importance for {best_model_name} ({importance_type})')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()

fig_path = os.path.join(OUTPUT_DIR, 'feature_importance_top10.png')
plt.savefig(fig_path)
plt.close()
print(f"✅ Top 10 Feature Importance plot saved to: {fig_path}")

# --- 7. SUMMARY ---
print("\n📋 7. SUMMARY AND RECOMMENDATIONS")
print("=" * 50)
print(f"✅ Analysis completed on {X.shape[0]} data records from {unique_wells} unique locations.")
print(f"🏆 Best performing model: {best_model_name}")
print(f"📊 Best model performance (Test Set):")
print(f"   - MAPE: {results_df.loc[best_model_name, 'Test_MAPE']:.2f}%")
print(f"   - R² Score: {results_df.loc[best_model_name, 'Test_R²']:.3f}")
print(f"   - RMSE: {results_df.loc[best_model_name, 'Test_RMSE']:.3f} meters")

print(f"\n🎯 Top 5 most important features for {best_model_name}:")
for i, feature in enumerate(feature_importance_df['Feature'].head(5)):
    print(f"   {i+1}. {feature}")

# --- FINAL ---
end_time = time.time()
execution_time = (end_time - start_time)
print(f"\n⏱️ Total execution time: {execution_time:.2f} seconds ({execution_time/60:.2f} minutes)")
print(f"\n🎉 Analysis successful! All results are saved in the '{OUTPUT_DIR}' folder.")

#### The code to focus on the best performing models (Extra Trees, Random Forest, and Decision Tree) and implement the three different data splitting scenarios

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
import time
from pathlib import Path
import os
from datetime import datetime

warnings.filterwarnings('ignore')

# Set professional plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10

class GroundwaterPredictor:
    """Professional groundwater depth prediction analysis with multiple splitting strategies."""
    
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None
        self.df_filtered = None
        self.results = {}
        self.feature_cols = []
        self.target_col = 'Depth to water (m)'
        self.models = {
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            'Decision Tree': DecisionTreeRegressor(random_state=42)
        }
        
        # Timing variables
        self.start_time = None
        self.timing_log = []
        
        # Create a unique output directory for each run
        run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.output_dir = Path(f"groundwater_analysis_results_{run_timestamp}")
        self.output_dir.mkdir(exist_ok=True)
        print(f"📂 Output will be saved to: {self.output_dir}")
        
    def log_time(self, step_name):
        """Log timing information for each step."""
        current_time = time.time()
        if self.start_time is None:
            self.start_time = current_time
            elapsed = 0
        else:
            elapsed = current_time - self.start_time
        
        self.timing_log.append({
            'step': step_name,
            'elapsed_minutes': elapsed / 60,
            'timestamp': datetime.now().strftime('%H:%M:%S')
        })
        
        print(f"⏱️  {step_name} completed at {datetime.now().strftime('%H:%M:%S')} "
              f"(Total elapsed: {elapsed/60:.2f} minutes)")
        
    def print_timing_summary(self):
        """Print a summary of timing information."""
        if not self.timing_log:
            return
            
        print("\n" + "="*60)
        print("⏰ TIMING SUMMARY")
        print("="*60)
        
        for i, entry in enumerate(self.timing_log):
            if i == 0:
                step_time = entry['elapsed_minutes']
            else:
                step_time = entry['elapsed_minutes'] - self.timing_log[i-1]['elapsed_minutes']
            
            print(f"{entry['step']:<40} | {step_time:>8.2f} min | {entry['timestamp']}")
        
        total_time = self.timing_log[-1]['elapsed_minutes']
        print("-" * 60)
        print(f"{'TOTAL RUNTIME':<40} | {total_time:>8.2f} min")
        print("="*60)
    
    def mean_absolute_percentage_error(self, y_true, y_pred):
        """Calculate Mean Absolute Percentage Error (MAPE), handling zero values in y_true."""
        y_true, y_pred = np.array(y_true), np.array(y_pred)
        # Avoid division by zero
        non_zero_mask = y_true != 0
        if not np.any(non_zero_mask):
            return 0.0
        return np.mean(np.abs((y_true[non_zero_mask] - y_pred[non_zero_mask]) / y_true[non_zero_mask])) * 100
    
    def load_and_preprocess_data(self):
        """Load and preprocess the dataset."""
        print("🌊 Chile Groundwater Depth Prediction Analysis")
        print("=" * 60)
        print(f"🚀 Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("\n📊 Loading and preprocessing dataset...")
        
        self.log_time("Analysis Started")
        
        try:
            if str(self.file_path).lower().endswith('.csv'):
                self.df = pd.read_csv(self.file_path)
            elif os.path.exists(self.file_path):
                self.df = pd.read_excel(self.file_path)
            else:
                raise FileNotFoundError(f"File not found at {self.file_path}")
            print(f"✅ Dataset loaded successfully! Shape: {self.df.shape}")
        except Exception as e:
            print(f"❌ Error loading file: {e}")
            return False
        
        self.log_time("Data Loading")
        
        print(f"Original dataset shape: {self.df.shape}")
        self.df_filtered = self.df[self.df['Status'].isnull()].copy()
        print(f"Filtered dataset shape (Status is blank): {self.df_filtered.shape}")
        
        if len(self.df_filtered) == 0:
            print("❌ No rows with blank Status found! Using all data for demonstration.")
            self.df_filtered = self.df.copy()
        
        initial_rows = len(self.df_filtered)
        self.df_filtered = self.df_filtered.dropna(subset=[self.target_col])
        print(f"Removed {initial_rows - len(self.df_filtered)} rows with missing target values")
        
        self._prepare_features()
        self.log_time("Data Preprocessing")
        return True
        
    def _prepare_features(self):
        """Prepare feature matrix and target variable."""
        numeric_cols = self.df_filtered.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = self.df_filtered.select_dtypes(include=['object']).columns.tolist()
        
        if self.target_col in numeric_cols:
            numeric_cols.remove(self.target_col)
        
        # These columns are used for splitting but are not model features
        columns_to_exclude_from_features = ['Status', 'Code', 'Date_String']
        for col in columns_to_exclude_from_features:
            if col in categorical_cols:
                categorical_cols.remove(col)
            if col in numeric_cols:
                numeric_cols.remove(col)
        
        for col in numeric_cols:
            if self.df_filtered[col].isnull().sum() > 0:
                self.df_filtered[col].fillna(self.df_filtered[col].median(), inplace=True)
        
        for col in categorical_cols:
            if col in self.df_filtered.columns:
                if self.df_filtered[col].isnull().sum() > 0:
                    self.df_filtered[col].fillna(self.df_filtered[col].mode()[0], inplace=True)
                le = LabelEncoder()
                self.df_filtered[col + '_encoded'] = le.fit_transform(self.df_filtered[col].astype(str))
        
        encoded_categorical_cols = [col + '_encoded' for col in categorical_cols if col in self.df_filtered.columns]
        self.feature_cols = numeric_cols + encoded_categorical_cols
        print(f"Final feature set: {len(self.feature_cols)} features")
        
    def run_analysis(self):
        """Run complete analysis with four splitting strategies."""
        if self.df_filtered is None or len(self.df_filtered) == 0:
            print("❌ Data not loaded or empty. Aborting analysis.")
            return

        X = self.df_filtered[self.feature_cols].values
        y = self.df_filtered[self.target_col].values
        
        # Define all four splitting strategies
        splitting_strategies = {
            'Random Split': lambda: train_test_split(X, y, test_size=0.3, random_state=42),
            'Well ID Split': lambda: self._well_id_split(X, y),
            'Spatial Grid Split': lambda: self.create_spatial_grid_split(X, y, test_size=0.3),
            'Temporal Well Split': lambda: self.create_temporal_well_split(X, y, train_ratio=0.7)
        }
        
        all_results = {}
        
        for strategy_name, split_func in splitting_strategies.items():
            print(f"\n{'='*60}")
            print(f"🔄 Analysis with {strategy_name}")
            print(f"{'='*60}")
            
            try:
                X_train, X_test, y_train, y_test = split_func()
                
                if len(X_train) == 0 or len(X_test) == 0:
                    print(f"  ⚠️  Warning: {strategy_name} resulted in an empty train or test set. Skipping.")
                    continue

                print(f"Training samples: {len(X_train)} ({len(X_train)/(len(X_train)+len(X_test))*100:.1f}%)")
                print(f"Testing samples: {len(X_test)} ({len(X_test)/(len(X_train)+len(X_test))*100:.1f}%)")
                
                strategy_results = self._train_and_evaluate_models(X_train, X_test, y_train, y_test)
                all_results[strategy_name] = strategy_results
                
                self._generate_plots_for_strategy(strategy_results, y_train, y_test, strategy_name)
                
            except Exception as e:
                print(f"  ❌ ERROR during {strategy_name}: {e}")
                
            self.log_time(f"{strategy_name} Analysis")
        
        self._create_final_comparison_plots(all_results)
        self.log_time("Final Comparison Plots")
        
        self._save_results_to_excel(all_results)
        self.log_time("Excel Export")
        
        self.print_timing_summary()
        print(f"\n🎉 Analysis completed! Results saved in '{self.output_dir}' directory")
        
    def _well_id_split(self, X, y):
        """Split data based on Well ID ('Code' column)."""
        if 'Code' not in self.df_filtered.columns:
            print("Warning: 'Code' column not found for Well ID split. Using random split instead.")
            return train_test_split(X, y, test_size=0.3, random_state=42)
        
        unique_wells = self.df_filtered['Code'].unique()
        train_wells, test_wells = train_test_split(unique_wells, test_size=0.3, random_state=42)
        
        train_mask = self.df_filtered['Code'].isin(train_wells)
        test_mask = self.df_filtered['Code'].isin(test_wells)
        
        return X[train_mask], X[test_mask], y[train_mask], y[test_mask]

    def create_spatial_grid_split(self, X, y, test_size=0.3, n_grid_x=10, n_grid_y=10):
        """Create spatial grid-based train/test split."""
        coord_cols = ['Longitude_GCS_WGS_1984', 'Latitude_GCS_WGS_1984']
        if not all(col in self.df_filtered.columns for col in coord_cols):
            print("Warning: Spatial coordinate columns not found for Spatial Grid split. Using random split instead.")
            return train_test_split(X, y, test_size=test_size, random_state=42)
        
        coords = self.df_filtered[coord_cols].values
        lon_edges = np.linspace(coords[:, 0].min(), coords[:, 0].max(), n_grid_x + 1)
        lat_edges = np.linspace(coords[:, 1].min(), coords[:, 1].max(), n_grid_y + 1)
        
        lon_bins = np.clip(np.digitize(coords[:, 0], lon_edges) - 1, 0, n_grid_x - 1)
        lat_bins = np.clip(np.digitize(coords[:, 1], lat_edges) - 1, 0, n_grid_y - 1)
        grid_ids = lon_bins * n_grid_y + lat_bins
        
        unique_grids = np.unique(grid_ids)
        train_grids, test_grids = train_test_split(unique_grids, test_size=test_size, random_state=42)
        
        train_mask = np.isin(grid_ids, train_grids)
        test_mask = np.isin(grid_ids, test_grids)
        
        return X[train_mask], X[test_mask], y[train_mask], y[test_mask]

    def create_temporal_well_split(self, X, y, train_ratio=0.7):
        """Split data temporally within each well."""
        if 'Code' not in self.df_filtered.columns or 'Date_String' not in self.df_filtered.columns:
            print("Warning: 'Code' or 'Date_String' column not found for Temporal Well split. Using random split instead.")
            return train_test_split(X, y, test_size=1-train_ratio, random_state=42)

        train_indices, test_indices = [], []
        
        # Ensure Date_String is sortable
        df_temp = self.df_filtered.copy()
        df_temp['Date_String'] = pd.to_datetime(df_temp['Date_String'], errors='coerce')
        df_temp.dropna(subset=['Date_String'], inplace=True)
        
        for _, group in df_temp.groupby('Code'):
            group_sorted = group.sort_values('Date_String')
            indices = group_sorted.index.tolist()
            n_records = len(indices)
            if n_records < 2:
                train_indices.extend(indices)
            else:
                n_train = max(1, int(n_records * train_ratio))
                train_indices.extend(indices[:n_train])
                test_indices.extend(indices[n_train:])
        
        train_mask = self.df_filtered.index.isin(train_indices)
        test_mask = self.df_filtered.index.isin(test_indices)
        
        return X[train_mask], X[test_mask], y[train_mask], y[test_mask]

    def _train_and_evaluate_models(self, X_train, X_test, y_train, y_test):
        """Train and evaluate all models."""
        results = {}
        for name, model in self.models.items():
            model_start_time = time.time()
            print(f"  Training {name}...")
            
            model.fit(X_train, y_train)
            
            y_pred_train = model.predict(X_train)
            y_pred_test = model.predict(X_test)
            
            model_time = (time.time() - model_start_time) / 60
            print(f"    ✅ {name} completed in {model_time:.2f} minutes")
            
            results[name] = {
                'model': model,
                'train_metrics': self._calculate_metrics(y_train, y_pred_train),
                'test_metrics': self._calculate_metrics(y_test, y_pred_test),
                'train_predictions': y_pred_train,
                'test_predictions': y_pred_test,
                'training_time_minutes': model_time
            }
        return results
    
    def _calculate_metrics(self, y_true, y_pred):
        """Calculate performance metrics."""
        return {
            'r2': r2_score(y_true, y_pred),
            'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
            'mae': mean_absolute_error(y_true, y_pred),
            'mape': self.mean_absolute_percentage_error(y_true, y_pred)
        }

    def _generate_plots_for_strategy(self, strategy_results, y_train, y_test, strategy_name):
        """Generate all plots for a given modeling strategy."""
        print(f"  Generating plots for {strategy_name}...")
        for model_name, results in strategy_results.items():
            self._plot_predicted_vs_actual(y_test, results['test_predictions'], results, model_name, strategy_name, 'Testing')
            self._plot_predicted_vs_actual(y_train, results['train_predictions'], results, model_name, strategy_name, 'Training')
        self._create_feature_importance_plot(strategy_results, strategy_name)

    def _plot_predicted_vs_actual(self, y_true, y_pred, results_dict, model_name, strategy_name, data_split_name):
        """Creates and saves a single 'Predicted vs Actual' scatter plot."""
        plt.figure(figsize=(8, 8))
        metrics = results_dict['train_metrics'] if data_split_name == 'Training' else results_dict['test_metrics']
        
        ax = sns.scatterplot(x=y_true, y=y_pred, alpha=0.5, color='red', edgecolor='w', s=40)
        lims = [min(min(y_true), min(y_pred)), max(max(y_true), max(y_pred))]
        ax.plot(lims, lims, 'r--', lw=2, label='1:1 Line')
        ax.set_xlim(lims); ax.set_ylim(lims)
        
        ax.set_xlabel(f'Actual Depth to Water (m) - {data_split_name}')
        ax.set_ylabel(f'Predicted Depth to Water (m) - {data_split_name}')
        ax.set_title(f'{model_name}: Predicted vs Actual\nStrategy: {strategy_name}')
        
        text_str = (f"R² = {metrics['r2']:.3f}\nRMSE = {metrics['rmse']:.2f} m\n"
                    f"MAPE = {metrics['mape']:.2f}%\nN = {len(y_true)}")
        props = dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='black')
        ax.text(0.05, 0.95, text_str, transform=ax.transAxes, fontsize=10, verticalalignment='top', bbox=props)
        
        filename = f'{strategy_name.lower().replace(" ", "_")}_{model_name.lower().replace(" ", "_")}_pred_vs_actual_{data_split_name.lower()}.png'
        plt.savefig(self.output_dir / filename, bbox_inches='tight', facecolor='white')
        plt.close()

    def _create_feature_importance_plot(self, results, strategy_name):
        """Create feature importance visualization."""
        num_models = len(results)
        fig, axes = plt.subplots(1, num_models, figsize=(7 * num_models, 8), sharex=True)
        if num_models == 1: axes = [axes]
        fig.suptitle(f'Feature Importance Analysis - {strategy_name}', fontsize=16, fontweight='bold')
        
        for i, (name, res) in enumerate(results.items()):
            model = res['model']
            if hasattr(model, 'feature_importances_'):
                importances = pd.Series(model.feature_importances_, index=self.feature_cols).nlargest(15)
                importances.plot(kind='barh', ax=axes[i], color=sns.color_palette("viridis", num_models)[i])
                axes[i].set_title(name)
                axes[i].invert_yaxis()
        
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.savefig(self.output_dir / f'feature_importance_{strategy_name.lower().replace(" ", "_")}.png', bbox_inches='tight', facecolor='white')
        plt.close()
    
    def _create_final_comparison_plots(self, all_results):
        """Creates summary comparison plots for both testing and training sets."""
        if not all_results:
            print("No results to plot. Skipping final comparison.")
            return
        print("  Generating final summary plots...")
        self._generate_strategy_comparison_figure(all_results, 'Testing')
        self._generate_strategy_comparison_figure(all_results, 'Training')

    def _generate_strategy_comparison_figure(self, all_results, data_split):
        """Generates a 2x2 figure comparing performance across strategies for a given data split."""
        data_split_lower = data_split.lower()
        fig, axes = plt.subplots(2, 2, figsize=(18, 14))
        fig.suptitle(f'Performance Comparison Across Splitting Strategies ({data_split} Set)', fontsize=18, fontweight='bold')
        
        strategies = list(all_results.keys())
        models = list(self.models.keys())
        
        metrics_config = {
            'Training': {
                'R²': ('train_metrics', 'r2'), 'RMSE (m)': ('train_metrics', 'rmse'),
                'MAPE (%)': ('train_metrics', 'mape'), 'Training Time (min)': (None, 'training_time_minutes')
            },
            'Testing': {
                'R²': ('test_metrics', 'r2'), 'RMSE (m)': ('test_metrics', 'rmse'),
                'MAPE (%)': ('test_metrics', 'mape'), 'Training Time (min)': (None, 'training_time_minutes')
            }
        }
        metrics = metrics_config[data_split]
        flat_axes = axes.flatten()

        for i, (title, (metric_group, key)) in enumerate(metrics.items()):
            ax = flat_axes[i]
            width = 0.8 / len(strategies)
            x = np.arange(len(models))
            
            for j, strategy in enumerate(strategies):
                if metric_group is None:
                    values = [all_results[strategy][model][key] for model in models]
                else:
                    values = [all_results[strategy][model][metric_group][key] for model in models]
                
                offset = width * j - width * (len(strategies) - 1) / 2
                ax.bar(x + offset, values, width, label=strategy, alpha=0.8)
            
            ax.set_ylabel(f'{data_split} {title}')
            ax.set_title(f'{data_split} {title}')
            ax.set_xticks(x)
            ax.set_xticklabels(models, rotation=15, ha='right')
            ax.legend(title="Strategy")
            if 'R²' in title: ax.set_ylim(bottom=min(0, ax.get_ylim()[0] * 1.1), top=1.05)
            else: ax.set_ylim(bottom=0)
            ax.grid(axis='x', linestyle='--', alpha=0.6)
        
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        filename = f'strategy_comparison_summary_{data_split_lower}.png'
        plt.savefig(self.output_dir / filename, bbox_inches='tight', facecolor='white')
        print(f"    ✔️  Saved summary plot: {filename}")
        plt.close()

    def _save_results_to_excel(self, all_results):
        """Saves a detailed summary of all results to an Excel file."""
        if not all_results:
            return
        print("\n💾 Saving detailed results to Excel...")
        summary_list = []
        for strategy, models in all_results.items():
            for model_name, data in models.items():
                summary_list.append({
                    'Strategy': strategy, 'Model': model_name,
                    'Training Time (min)': data['training_time_minutes'],
                    'Train R2': data['train_metrics']['r2'], 'Test R2': data['test_metrics']['r2'],
                    'Train RMSE': data['train_metrics']['rmse'], 'Test RMSE': data['test_metrics']['rmse'],
                    'Train MAE': data['train_metrics']['mae'], 'Test MAE': data['test_metrics']['mae'],
                    'Train MAPE (%)': data['train_metrics']['mape'], 'Test MAPE (%)': data['test_metrics']['mape'],
                })
        
        summary_df = pd.DataFrame(summary_list)
        excel_path = self.output_dir / "performance_summary.xlsx"
        try:
            summary_df.to_excel(excel_path, index=False, sheet_name='Performance Metrics')
            print(f"✅ Excel summary saved to {excel_path}")
        except Exception as e:
            print(f"❌ Could not save Excel file. Error: {e}")

# ==============================================================================
# MAIN EXECUTION BLOCK
# ==============================================================================
def main():
    # IMPORTANT: Update this path to your actual file location.
    # Using a raw string (r"...") is recommended on Windows to handle backslashes.
    file_path = r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\results\chile_2025\groundwater_chile_and_elevation_dataset_2025_with_GEE_7_2_25.xlsx"
    
    if not os.path.exists(file_path):
        print(f"❌ FATAL ERROR: Input file not found at '{file_path}'")
        print("Please update the 'file_path' variable in the main() function.")
        return

    # Initialize and run analysis
    predictor = GroundwaterPredictor(file_path)
    if predictor.load_and_preprocess_data():
        predictor.run_analysis()

if __name__ == "__main__":
    main()

### Map creation

In [ ]:
# ============================================================================
# ENHANCED GROUNDWATER DATA MAPPING SCRIPT FOR SCIENTIFIC PUBLICATION
#
# This script generates publication-quality maps for visualizing groundwater
# data splitting strategies, designed for ScienceDirect journals.
#
# Key improvements:
# - Professional cartographic design with coordinate grids
# - Enhanced color schemes following scientific standards
# - Improved legends and labeling
# - Corrected basemap integration with a functional inset map
# - Robust statistical information display
# - Scale bars and north arrows
# - Inset location map showing Chile in South America
# - Assembled comparison figure of all split strategies
#
# Requirements:
# pip install pandas openpyxl geopandas contextily scikit-learn matplotlib shapely geodatasets matplotlib-scalebar pyproj
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.image as mpimg
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import geopandas as gpd
import contextily as cx
from shapely.geometry import Point, box, Polygon
from sklearn.model_selection import train_test_split
from pathlib import Path
import warnings
import matplotlib.gridspec as gridspec
from matplotlib_scalebar.scalebar import ScaleBar
import geodatasets
from pyproj import Transformer

# --- Setup ---
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Professional scientific plotting style
plt.style.use('default')
plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.family': 'Arial',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'axes.linewidth': 1.0,
    'grid.linewidth': 0.5,
    'grid.alpha': 0.4,
    'axes.edgecolor': 'black',
    'axes.facecolor': 'white',
    'figure.facecolor': 'white',
    'savefig.facecolor': 'white',
    'text.color': 'black'
})

class EnhancedGroundwaterMapper:
    """
    Enhanced mapper for generating publication-quality groundwater data split visualizations.
    Designed for scientific journals with professional cartographic standards.
    """

    def __init__(self, file_path, output_dir="publication_maps"):
        self.file_path = Path(file_path)
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(exist_ok=True)
        self.df = None
        self.gdf = None

        # Enhanced color scheme for scientific publications
        self.colors = {
            'all_data': '#2E86AB',      # Professional blue
            'train': '#A23B72',         # Deep magenta
            'test': '#F18F01',          # Orange
            'mixed': '#C73E1D',         # Red
            'background': '#F5F5F5',    # Light gray
            'text': '#2C3E50',          # Dark blue-gray
            'grid': '#CCCCCC',          # Light gray
            'coastline': '#34495E',     # Dark gray
            'border': '#7F8C8D'         # Medium gray
        }

        # Map styling parameters
        self.style = {
            'point_size': 40,
            'alpha': 0.85,
            'edge_color': 'white',
            'edge_width': 0.6,
            'mixed_marker': 's',  # Square for mixed points
            'figsize': (14, 16),
            'basemap_providers': {
                'light': cx.providers.CartoDB.Positron,
                'terrain': cx.providers.OpenTopoMap,
                'satellite': cx.providers.Esri.WorldImagery,
                'topo': cx.providers.OpenTopoMap
            },
            'default_basemap': 'terrain'  # Changed to terrain for groundwater context
        }

    def load_and_prepare_data(self):
        """Load and prepare groundwater data with enhanced error handling."""
        print("🌊 Loading Groundwater Dataset...")
        print("=" * 60)

        try:
            if not self.file_path.exists():
                raise FileNotFoundError(f"File not found: {self.file_path}")

            # Load data based on file extension
            if self.file_path.suffix.lower() == '.csv':
                self.df = pd.read_csv(self.file_path)
            elif self.file_path.suffix.lower() in ['.xlsx', '.xls']:
                self.df = pd.read_excel(self.file_path)
            else:
                raise ValueError(f"Unsupported file format: {self.file_path.suffix}")

            print(f"✅ Dataset loaded successfully! Shape: {self.df.shape}")
            
            # Convert Date column to datetime if it exists
            if 'Date' in self.df.columns:
                try:
                    self.df['Date'] = pd.to_datetime(self.df['Date'], errors='coerce')
                    print(f"📅 Date column converted to datetime format")
                except Exception as e:
                    print(f"⚠️ Could not convert Date column: {e}")

        except Exception as e:
            print(f"⚠️ Error loading file: {e}")
            print("📊 Creating demonstration dataset...")
            self._create_enhanced_demo_data()

        # Enhanced data preparation
        self._prepare_geodataframe()
        return True

    def _create_enhanced_demo_data(self):
        """Create enhanced demo dataset with realistic Chilean groundwater patterns."""
        print("🗺️ Generating realistic Chilean groundwater dataset...")

        # Create simplified Chile geometry
        chile_coords = [
            [-75, -18], [-70, -18], [-70, -22], [-68, -22], [-68, -27],
            [-70, -27], [-70, -33], [-73, -33], [-73, -42], [-75, -42],
            [-75, -50], [-72, -50], [-72, -56], [-68, -56], [-68, -53],
            [-75, -53], [-75, -18]
        ]
        chile_geom = Polygon(chile_coords)

        # Create realistic well distribution
        n_points = 1200
        n_wells = 600

        # Generate points with realistic clustering patterns
        points = []
        clusters = [
            {'center': (-70.0, -24.0), 'radius': 2.0, 'n_points': 200},
            {'center': (-70.7, -33.5), 'radius': 3.0, 'n_points': 600},
            {'center': (-72.0, -41.0), 'radius': 2.5, 'n_points': 400}
        ]

        for cluster in clusters:
            center_x, center_y = cluster['center']
            for _ in range(cluster['n_points']):
                angle = np.random.uniform(0, 2 * np.pi)
                distance = np.random.exponential(cluster['radius'] / 3)
                x = center_x + distance * np.cos(angle)
                y = center_y + distance * np.sin(angle)

                point = Point(x, y)
                if chile_geom.contains(point):
                    points.append(point)
                    if len(points) >= n_points:
                        break

        # Create well codes and temporal data
        well_codes = [f'CHL-{i:04d}' for i in range(n_wells)]
        well_codes.extend(np.random.choice(well_codes, n_points - n_wells, replace=True))
        np.random.shuffle(well_codes)

        # Generate realistic dates (2000-2024)
        start_date = pd.to_datetime('2000-01-01')
        end_date = pd.to_datetime('2024-12-31')
        date_range = (end_date - start_date).days
        dates = [start_date + pd.Timedelta(days=np.random.randint(0, date_range)) for _ in range(n_points)]

        # Create realistic depth values
        depths = np.random.lognormal(mean=2.5, sigma=0.8, size=n_points)
        depths = np.clip(depths, 0.5, 150)

        self.df = pd.DataFrame({
            'Code': well_codes,
            'Date': dates,
            'Depth to water (m)': depths,
            'Longitude_GCS_WGS_1984': [p.x for p in points],
            'Latitude_GCS_WGS_1984': [p.y for p in points],
            'Status': np.nan,
            'Region': np.random.choice(['Norte', 'Centro', 'Sur'], n_points)
        })

        print(f"📍 Demo dataset created: {len(self.df)} records, {n_wells} wells")

    def _prepare_geodataframe(self):
        """Prepare and clean the geodataframe."""
        df_clean = self.df.copy()

        if 'Status' in df_clean.columns:
            df_clean = df_clean[df_clean['Status'].isnull()].copy()

        if len(df_clean) == 0:
            print("⚠️ No records with blank Status. Using all data.")
            df_clean = self.df.copy()

        required_cols = ['Longitude_GCS_WGS_1984', 'Latitude_GCS_WGS_1984', 'Depth to water (m)']
        df_clean = df_clean.dropna(subset=required_cols).reset_index(drop=True)

        self.gdf = gpd.GeoDataFrame(
            df_clean,
            geometry=gpd.points_from_xy(df_clean['Longitude_GCS_WGS_1984'], df_clean['Latitude_GCS_WGS_1984']),
            crs="EPSG:4326"
        )
        print(f"🌍 GeoDataFrame prepared: {len(self.gdf)} valid records")

    def _add_north_arrow(self, ax, x=0.95, y=0.95, size=0.03, color='black'):
        """Add a north arrow to the map."""
        arrow = FancyArrowPatch((x, y - size/2), (x, y + size/2),
                               transform=ax.transAxes, arrowstyle='-|>',
                               mutation_scale=20, linewidth=2, color=color)
        ax.add_patch(arrow)
        ax.text(x, y + size + 0.01, 'N', transform=ax.transAxes,
                fontsize=12, fontweight='bold', ha='center', va='bottom')

    def _add_scale_bar(self, ax, gdf_wm):
        """Add a scale bar to the map."""
        scalebar = ScaleBar(1, 'm', length_fraction=0.25, location='lower left',
                           pad=0.5, color='black', frameon=True, box_alpha=0.8,
                           font_properties={'size': 9})
        ax.add_artist(scalebar)

    def _add_coordinate_grid(self, ax, gdf_wm):
        """Add coordinate grid with proper labels to a Mercator map."""
        bounds = gdf_wm.total_bounds
        x_min, y_min, x_max, y_max = bounds
        transformer = Transformer.from_crs('EPSG:3857', 'EPSG:4326', always_xy=True)

        def format_coord(val, is_longitude=True):
            direction = 'W' if is_longitude and val < 0 else 'E' if is_longitude and val > 0 else ''
            if not is_longitude:
                direction = 'S' if val < 0 else 'N' if val > 0 else ''
            return f"{abs(val):.1f}°{direction}"

        ax.grid(True, alpha=0.3, linewidth=0.5, color=self.colors['grid'], zorder=0)

        n_ticks = 5
        x_ticks = np.linspace(x_min, x_max, n_ticks)
        y_ticks = np.linspace(y_min, y_max, n_ticks)
        
        x_labels = [format_coord(lon, True) for lon, lat in transformer.itransform(zip(x_ticks, [y_min] * n_ticks))]
        y_labels = [format_coord(lat, False) for lon, lat in transformer.itransform(zip([x_min] * n_ticks, y_ticks))]

        ax.set_xticks(x_ticks)
        ax.set_yticks(y_ticks)
        ax.set_xticklabels(x_labels, fontsize=8)
        ax.set_yticklabels(y_labels, fontsize=8)

        ax.set_xlabel('Longitude', fontsize=10, fontweight='bold')
        ax.set_ylabel('Latitude', fontsize=10, fontweight='bold')

    def _add_inset_map(self, ax, gdf):
        """Add an inset map showing Chile's location in South America."""
        ax_inset = inset_axes(ax, width="30%", height="40%", loc='lower right',
                             bbox_to_anchor=(0, 0.05, 1, 1), bbox_transform=ax.transAxes,
                             borderpad=3, axes_class=plt.Axes)

        try:
            world = gpd.read_file(geodatasets.get_path('naturalearth.land'))
            # Filter for South America region (approximate bounds)
            south_america = world.cx[-85:-30, -60:15]
            
            # Create simplified Chile geometry for inset
            chile_coords = [
                [-75, -18], [-70, -18], [-70, -22], [-68, -22], [-68, -27],
                [-70, -27], [-70, -33], [-73, -33], [-73, -42], [-75, -42],
                [-75, -50], [-72, -50], [-72, -56], [-68, -56], [-68, -53],
                [-75, -53], [-75, -18]
            ]
            chile_geom = Polygon(chile_coords)
            chile = gpd.GeoDataFrame([1], geometry=[chile_geom], crs='EPSG:4326')
            
            south_america.plot(ax=ax_inset, color='lightgray', edgecolor='white', linewidth=0.5)
            chile.plot(ax=ax_inset, color=self.colors['all_data'], edgecolor='black', linewidth=1)
            
        except Exception as e:
            print(f"⚠️ Could not load world data for inset: {e}")
            # Create a simple fallback inset
            # Draw simplified South America outline
            sa_coords = [(-85, -60), (-30, -60), (-30, 15), (-85, 15), (-85, -60)]
            sa_x, sa_y = zip(*sa_coords)
            ax_inset.plot(sa_x, sa_y, color='lightgray', linewidth=2)
            ax_inset.fill(sa_x, sa_y, color='lightgray', alpha=0.3)
            
            # Draw simplified Chile
            chile_coords = [
                [-75, -18], [-70, -18], [-70, -22], [-68, -22], [-68, -27],
                [-70, -27], [-70, -33], [-73, -33], [-73, -42], [-75, -42],
                [-75, -50], [-72, -50], [-72, -56], [-68, -56], [-68, -53],
                [-75, -53], [-75, -18]
            ]
            chile_x, chile_y = zip(*chile_coords)
            ax_inset.plot(chile_x, chile_y, color='black', linewidth=1)
            ax_inset.fill(chile_x, chile_y, color=self.colors['all_data'], alpha=0.7)

        # Add study area box
        bounds = gdf.total_bounds
        study_box = box(bounds[0], bounds[1], bounds[2], bounds[3])
        gpd.GeoDataFrame([1], geometry=[study_box], crs=gdf.crs).plot(
            ax=ax_inset, facecolor='none', edgecolor='red', linewidth=2
        )

        ax_inset.set_xlim(-85, -30)
        ax_inset.set_ylim(-60, 15)
        ax_inset.set_xticks([])
        ax_inset.set_yticks([])
        ax_inset.set_title('Study Area', fontsize=8, pad=2)

        for spine in ax_inset.spines.values():
            spine.set_edgecolor('black')
            spine.set_linewidth(1)

    def _create_enhanced_basemap(self, ax, gdf_wm, basemap_style='terrain'):
        """Create enhanced basemap with professional styling."""
        try:
            provider = self.style['basemap_providers'].get(
                basemap_style, self.style['basemap_providers'][self.style['default_basemap']]
            )
            cx.add_basemap(ax, crs=gdf_wm.crs.to_string(), source=provider, zoom='auto', alpha=0.7)
        except Exception as e:
            print(f"⚠️ Could not add basemap: {e}")
            # Set a simple background color instead
            ax.set_facecolor('#F0F0F0')
            
        ax.set_aspect('equal')
        self._add_coordinate_grid(ax, gdf_wm)
        return ax

    def _add_map_elements(self, ax, title, gdf_wm, gdf_original, legend_patches=None):
        """Add professional map elements including scale bar, north arrow, and stats."""
        title_text = ax.set_title(title, fontsize=14, fontweight='bold', color=self.colors['text'], pad=20)
        title_text.set_bbox(dict(facecolor='white', alpha=0.8, edgecolor='none', pad=5))

        if legend_patches:
            legend = ax.legend(handles=legend_patches, loc='upper left', frameon=True, fancybox=True,
                             shadow=True, fontsize=9, title='Data Split', title_fontsize=10)
            frame = legend.get_frame()
            frame.set_facecolor('white')
            frame.set_alpha(0.9)
            frame.set_edgecolor('gray')

        self._add_north_arrow(ax)
        self._add_scale_bar(ax, gdf_wm)
        self._add_inset_map(ax, gdf_original)

        # Improved and robust statistics box
        stats_text = f"Total Records: {len(gdf_wm):,}"
        if 'split_color' in gdf_wm.columns:
            n_train = (gdf_wm['split_color'] == self.colors['train']).sum()
            n_test = (gdf_wm['split_color'] == self.colors['test']).sum()
            n_mixed = (gdf_wm['split_color'] == self.colors['mixed']).sum()
            total_split = n_train + n_test + n_mixed
            if total_split > 0:
                if n_train > 0: stats_text += f"\nTraining: {n_train:,} ({n_train/total_split*100:.1f}%)"
                if n_test > 0: stats_text += f"\nTesting: {n_test:,} ({n_test/total_split*100:.1f}%)"
                if n_mixed > 0: stats_text += f"\nMixed Recs: {n_mixed:,} ({n_mixed/total_split*100:.1f}%)"

        props = dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='gray', alpha=0.9, linewidth=1)
        ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, fontsize=9,
                verticalalignment='top', bbox=props)

        ax.text(0.5, -0.05, "Data Source: Chilean Groundwater Monitoring Network (DGA)",
                transform=ax.transAxes, fontsize=8, ha='center', style='italic', color='gray')

        return ax

    def _plot_enhanced_map(self, gdf, title, filename, color_column=None,
                          single_color=None, legend_patches=None, basemap_style='terrain'):
        """Create and save an enhanced publication-quality map."""
        fig, ax = plt.subplots(figsize=self.style['figsize'], dpi=300)
        fig.patch.set_facecolor('white')

        gdf_wm = gdf.to_crs(epsg=3857)
        self._create_enhanced_basemap(ax, gdf_wm, basemap_style)

        if single_color:
            gdf_wm.plot(ax=ax, color=single_color, markersize=self.style['point_size'],
                       alpha=self.style['alpha'], edgecolor=self.style['edge_color'],
                       linewidth=self.style['edge_width'], zorder=5)
        elif color_column and color_column in gdf_wm.columns:
            for color, marker in [(self.colors['train'], 'o'), (self.colors['test'], 'o'), (self.colors['mixed'], 's')]:
                subset = gdf_wm[gdf_wm[color_column] == color]
                if not subset.empty:
                    subset.plot(ax=ax, color=color, marker=marker, markersize=self.style['point_size'],
                               alpha=self.style['alpha'], edgecolor=self.style['edge_color'],
                               linewidth=self.style['edge_width'], zorder=5 if marker == 'o' else 6)

        self._add_map_elements(ax, title, gdf_wm, gdf, legend_patches)
        for spine in ax.spines.values():
            spine.set_edgecolor('black')

        filepath = self.output_dir / filename
        plt.savefig(filepath, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"    ✅ Saved: {filepath.name}")
        plt.close(fig)

    def generate_publication_maps(self):
        """Generate all five publication-quality maps and a comparison figure."""
        if self.gdf is None or self.gdf.empty:
            print("❌ No data available for mapping")
            return

        print("\n🗺️ Generating Publication-Quality Maps...")
        print("=" * 60)

        # Map 1: All Data Locations
        print("1. All Well Locations...")
        self._plot_enhanced_map(
            self.gdf, 'Groundwater Monitoring Well Distribution in Chile', 'Fig1_all_locations_enhanced.png',
            single_color=self.colors['all_data'],
            legend_patches=[mpatches.Patch(color=self.colors['all_data'], label=f'Well Records (n={len(self.gdf):,})')],
            basemap_style='light'
        )

        # Map 2: Random Split
        print("2. Random Split...")
        gdf_random = self.gdf.copy()
        train_idx, test_idx = train_test_split(gdf_random.index, test_size=0.3, random_state=42)
        gdf_random['split_color'] = self.colors['train']
        gdf_random.loc[test_idx, 'split_color'] = self.colors['test']
        mixed_wells = set(gdf_random.loc[train_idx, 'Code']).intersection(set(gdf_random.loc[test_idx, 'Code']))
        if mixed_wells:
            gdf_random.loc[gdf_random['Code'].isin(mixed_wells), 'split_color'] = self.colors['mixed']
        legend_patches = [
            mpatches.Patch(color=self.colors['train'], label=f'Training Set (n={len(train_idx):,})'),
            mpatches.Patch(color=self.colors['test'], label=f'Testing Set (n={len(test_idx):,})')
        ]
        if mixed_wells:
            legend_patches.append(mpatches.Patch(color=self.colors['mixed'], label=f'Mixed Wells (n={len(mixed_wells):,})'))
        self._plot_enhanced_map(
            gdf_random, 'Random Data Split Strategy', 'Fig2_random_split_enhanced.png',
            color_column='split_color', legend_patches=legend_patches, basemap_style='terrain'
        )

        # Map 3: Well ID Split
        print("3. Well ID Split...")
        unique_wells = self.gdf['Code'].unique()
        train_wells, test_wells = train_test_split(unique_wells, test_size=0.3, random_state=42)
        gdf_wells = self.gdf.copy()
        gdf_wells['split_color'] = np.where(gdf_wells['Code'].isin(train_wells), self.colors['train'], self.colors['test'])
        train_count = (gdf_wells['split_color'] == self.colors['train']).sum()
        test_count = len(gdf_wells) - train_count
        legend_patches = [
            mpatches.Patch(color=self.colors['train'], label=f'Training Wells ({len(train_wells):,} wells, {train_count:,} recs)'),
            mpatches.Patch(color=self.colors['test'], label=f'Testing Wells ({len(test_wells):,} wells, {test_count:,} recs)')
        ]
        self._plot_enhanced_map(
            gdf_wells, 'Well-Based Split Strategy (Spatially Independent)', 'Fig3_well_id_split_enhanced.png',
            color_column='split_color', legend_patches=legend_patches, basemap_style='terrain'
        )

        # Map 4: Spatial Grid Split
        print("4. Spatial Grid Split...")
        gdf_spatial = self.gdf.copy()
        lon_bins = pd.cut(gdf_spatial.geometry.x, bins=10, labels=False)
        lat_bins = pd.cut(gdf_spatial.geometry.y, bins=10, labels=False)
        test_mask = ((lon_bins + lat_bins) % 2 == 0)
        gdf_spatial['split_color'] = np.where(test_mask, self.colors['test'], self.colors['train'])
        train_count = (~test_mask).sum()
        test_count = test_mask.sum()
        legend_patches = [
            mpatches.Patch(color=self.colors['train'], label=f'Training Cells (n={train_count:,})'),
            mpatches.Patch(color=self.colors['test'], label=f'Testing Cells (n={test_count:,})')
        ]
        self._plot_enhanced_map(
            gdf_spatial, 'Spatial Grid Split Strategy (Checkerboard Pattern)', 'Fig4_spatial_grid_split_enhanced.png',
            color_column='split_color', legend_patches=legend_patches, basemap_style='light'
        )

        # Map 5: Temporal Split
        print("5. Temporal Split...")
        if 'Date' in self.gdf.columns and pd.api.types.is_datetime64_any_dtype(self.gdf['Date']):
            gdf_temporal = self.gdf.sort_values('Date').copy()
            split_date = gdf_temporal['Date'].quantile(0.7)
            gdf_temporal['split_color'] = np.where(gdf_temporal['Date'] <= split_date, self.colors['train'], self.colors['test'])

            train_wells = set(gdf_temporal[gdf_temporal['split_color'] == self.colors['train']]['Code'])
            test_wells = set(gdf_temporal[gdf_temporal['split_color'] == self.colors['test']]['Code'])
            mixed_wells = train_wells.intersection(test_wells)

            if mixed_wells:
                gdf_temporal.loc[gdf_temporal['Code'].isin(mixed_wells), 'split_color'] = self.colors['mixed']

            train_count = (gdf_temporal['split_color'] == self.colors['train']).sum()
            test_count = (gdf_temporal['split_color'] == self.colors['test']).sum()
            legend_patches = [
                mpatches.Patch(color=self.colors['train'], label=f'Training Period (Before {split_date.date()}, n={train_count:,})'),
                mpatches.Patch(color=self.colors['test'], label=f'Testing Period (After {split_date.date()}, n={test_count:,})')
            ]
            if mixed_wells:
                legend_patches.append(mpatches.Patch(color=self.colors['mixed'], label=f'Wells with Data in Both Periods (n={len(mixed_wells):,})'))

            self._plot_enhanced_map(
                gdf_temporal, f'Temporal Split Strategy (Split at {split_date.year})', 'Fig5_temporal_split_enhanced.png',
                color_column='split_color', legend_patches=legend_patches, basemap_style='terrain'
            )
        else:
            print("   ⚠️ No valid 'Date' column found. Creating placeholder for temporal split...")
            # Create a placeholder image for temporal split
            fig, ax = plt.subplots(figsize=(14, 16), dpi=300)
            ax.text(0.5, 0.5, "Temporal Split Not Available\n(No valid Date column in dataset)", 
                   transform=ax.transAxes, ha='center', va='center', fontsize=20,
                   bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.8))
            ax.set_xlim(0, 1)
            ax.set_ylim(0, 1)
            ax.axis('off')
            filepath = self.output_dir / 'Fig5_temporal_split_enhanced.png'
            plt.savefig(filepath, dpi=300, bbox_inches='tight', facecolor='white')
            plt.close(fig)

        self._generate_comparison_figure()

    def _generate_comparison_figure(self):
        """Generate a 2x2 comparison figure by assembling the saved split-strategy maps."""
        print("\n6. Generating Comparison Overview Figure...")

        strategies_to_plot = [
            {'filename': 'Fig2_random_split_enhanced.png', 'title': 'A) Random Split'},
            {'filename': 'Fig3_well_id_split_enhanced.png', 'title': 'B) Well-Based Split'},
            {'filename': 'Fig4_spatial_grid_split_enhanced.png', 'title': 'C) Spatial Grid Split'},
            {'filename': 'Fig5_temporal_split_enhanced.png', 'title': 'D) Temporal Split'}
        ]

        # Create a 2x2 subplot figure
        fig, axes = plt.subplots(2, 2, figsize=(20, 16), dpi=300)
        fig.patch.set_facecolor('white')
        fig.suptitle('Groundwater Data Splitting Strategies for Machine Learning', 
                    fontsize=16, fontweight='bold', y=0.95)

        for i, strategy in enumerate(strategies_to_plot):
            row, col = i // 2, i % 2
            ax = axes[row, col]
            
            image_path = self.output_dir / strategy['filename']
            
            try:
                if image_path.exists():
                    img = mpimg.imread(image_path)
                    ax.imshow(img)
                    ax.set_title(strategy['title'], fontsize=14, fontweight='bold', pad=10)
                else:
                    # If image doesn't exist, show placeholder
                    ax.text(0.5, 0.5, f"Image not found:\n{strategy['filename']}", 
                           transform=ax.transAxes, ha='center', va='center',
                           fontsize=12, bbox=dict(boxstyle='round', facecolor='lightgray'))
                    ax.set_title(strategy['title'], fontsize=14, fontweight='bold', pad=10)
            except Exception as e:
                ax.text(0.5, 0.5, f"Error loading image:\n{str(e)}", 
                       transform=ax.transAxes, ha='center', va='center',
                       fontsize=10, bbox=dict(boxstyle='round', facecolor='lightcoral'))
                ax.set_title(strategy['title'], fontsize=14, fontweight='bold', pad=10)
            
            ax.axis('off')

        plt.tight_layout()
        plt.subplots_adjust(top=0.92)
        
        comparison_path = self.output_dir / 'Fig6_comparison_all_strategies.png'
        plt.savefig(comparison_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"    ✅ Saved: {comparison_path.name}")
        plt.close(fig)

    def generate_summary_report(self):
        """Generate a comprehensive summary report of the mapping process."""
        print("\n📊 Generating Summary Report...")
        print("=" * 60)

        report = []
        report.append("# GROUNDWATER DATA MAPPING REPORT")
        report.append("=" * 50)
        report.append(f"Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
        report.append("")

        # Dataset Summary
        report.append("## DATASET SUMMARY")
        report.append(f"- Total Records: {len(self.gdf):,}")
        report.append(f"- Unique Wells: {self.gdf['Code'].nunique():,}")
        report.append(f"- Geographic Extent:")
        bounds = self.gdf.total_bounds
        report.append(f"  - Longitude: {bounds[0]:.3f}° to {bounds[2]:.3f}°")
        report.append(f"  - Latitude: {bounds[1]:.3f}° to {bounds[3]:.3f}°")
        
        if 'Date' in self.gdf.columns and pd.api.types.is_datetime64_any_dtype(self.gdf['Date']):
            valid_dates = self.gdf['Date'].dropna()
            if len(valid_dates) > 0:
                report.append(f"- Temporal Range: {valid_dates.min().date()} to {valid_dates.max().date()}")
        
        if 'Depth to water (m)' in self.gdf.columns:
            depth_stats = self.gdf['Depth to water (m)'].describe()
            report.append(f"- Depth Statistics (m):")
            report.append(f"  - Mean: {depth_stats['mean']:.2f}")
            report.append(f"  - Median: {depth_stats['50%']:.2f}")
            report.append(f"  - Range: {depth_stats['min']:.2f} to {depth_stats['max']:.2f}")
        
        report.append("")

        # Files Generated
        report.append("## FILES GENERATED")
        output_files = list(self.output_dir.glob("*.png"))
        for i, file in enumerate(sorted(output_files), 1):
            report.append(f"{i}. {file.name}")
        report.append("")

        # Technical Details
        report.append("## TECHNICAL DETAILS")
        report.append("- Coordinate System: WGS84 (EPSG:4326) → Web Mercator (EPSG:3857)")
        report.append("- Map Resolution: 300 DPI")
        report.append("- Basemap Source: OpenTopoMap / CartoDB")
        report.append("- Color Scheme: Scientific publication standard")
        report.append("")

        # Recommendations
        report.append("## RECOMMENDATIONS FOR MACHINE LEARNING")
        report.append("1. **Random Split**: Use for general model evaluation")
        report.append("2. **Well-Based Split**: Use for spatial generalization assessment")
        report.append("3. **Spatial Grid Split**: Use for testing spatial interpolation")
        report.append("4. **Temporal Split**: Use for time-series forecasting validation")
        report.append("")

        # Save report
        report_path = self.output_dir / 'mapping_report.txt'
        with open(report_path, 'w') as f:
            f.write('\n'.join(report))
        
        print(f"    ✅ Report saved: {report_path.name}")
        print("\n" + "\n".join(report))

def main():
    """Main execution function."""
    print("🌊 ENHANCED GROUNDWATER DATA MAPPING SYSTEM")
    print("=" * 60)
    print("Generating publication-quality maps for scientific journals")
    print("Compatible with ScienceDirect and other academic publications")
    print("=" * 60)

    # Initialize mapper
    mapper = EnhancedGroundwaterMapper(r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\results\chile_2025\groundwater_chile_and_elevation_dataset_2025_with_GEE_7_2_25.xlsx")

    # Load and prepare data
    if not mapper.load_and_prepare_data():
        print("❌ Failed to load data. Exiting.")
        return
    
    # Generate all maps
    mapper.generate_publication_maps()
    
    # Generate summary report
    mapper.generate_summary_report()
    
    print("\n🎉 SUCCESS! All maps generated successfully!")
    print("=" * 60)
    print(f"📁 Output directory: {mapper.output_dir}")
    print("📊 Files generated:")
    for file in sorted(mapper.output_dir.glob("*.png")):
        print(f"   - {file.name}")
    print(f"📄 Report: mapping_report.txt")
    print("\n✨ Maps are ready for publication!")

if __name__ == "__main__":
    main()

#### Temporal selection based on enough of data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance
import warnings
import time
from pathlib import Path
import os
from datetime import datetime
warnings.filterwarnings('ignore')

# Set professional plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 9

class GroundwaterPredictor:
    """Professional groundwater depth prediction analysis with improved temporal well split."""
    
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None
        self.df_filtered = None
        self.results = {}
        self.predictions = {}
        self.feature_cols = []
        self.target_col = 'Depth to water (m)'
        self.models = {
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            'Decision Tree': DecisionTreeRegressor(random_state=42)
        }
        
        # Timing variables
        self.start_time = None
        self.timing_log = []
        
        # Create output directory for figures
        self.output_dir = Path("groundwater_temporal_analysis_results")
        self.output_dir.mkdir(exist_ok=True)
        
    def log_time(self, step_name):
        """Log timing information for each step."""
        current_time = time.time()
        if self.start_time is None:
            self.start_time = current_time
            elapsed = 0
        else:
            elapsed = current_time - self.start_time
        
        self.timing_log.append({
            'step': step_name,
            'elapsed_minutes': elapsed / 60,
            'timestamp': datetime.now().strftime('%H:%M:%S')
        })
        
        print(f"⏱️  {step_name} completed at {datetime.now().strftime('%H:%M:%S')} "
              f"(Total elapsed: {elapsed/60:.2f} minutes)")
        
    def print_timing_summary(self):
        """Print a summary of timing information."""
        if not self.timing_log:
            return
            
        print("\n" + "="*60)
        print("⏰ TIMING SUMMARY")
        print("="*60)
        
        for i, entry in enumerate(self.timing_log):
            if i == 0:
                step_time = entry['elapsed_minutes']
            else:
                step_time = entry['elapsed_minutes'] - self.timing_log[i-1]['elapsed_minutes']
            
            print(f"{entry['step']:<40} | {step_time:>8.2f} min | {entry['timestamp']}")
        
        total_time = self.timing_log[-1]['elapsed_minutes']
        print("-" * 60)
        print(f"{'TOTAL RUNTIME':<40} | {total_time:>8.2f} min")
        print("="*60)
    
    def mean_absolute_percentage_error(self, y_true, y_pred):
        """Calculate Mean Absolute Percentage Error (MAPE)"""
        epsilon = 1e-10
        return np.mean(np.abs((y_true - y_pred) / (y_true + epsilon))) * 100
    
    def load_and_preprocess_data(self):
        """Load and preprocess the dataset."""
        print("🌊 Chile Groundwater Depth Prediction Analysis")
        print("📊 Using Improved Temporal Well Split Strategy")
        print("=" * 60)
        print(f"🚀 Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("\n📊 Loading and preprocessing dataset...")
        
        self.log_time("Analysis Started")
        
        try:
            if self.file_path.endswith('.csv'):
                self.df = pd.read_csv(self.file_path)
            else:
                self.df = pd.read_excel(self.file_path)
            print(f"✅ Dataset loaded successfully! Shape: {self.df.shape}")
        except Exception as e:
            print(f"❌ Error loading file: {e}")
            # Create sample data for demonstration
            print("Creating sample data for demonstration...")
            np.random.seed(42)
            n_samples = 1000
            
            # Create sample data with multiple records per well
            n_wells = 200
            records_per_well = np.random.poisson(5, n_wells) + 1  # At least 1 record per well
            
            well_data = []
            for i in range(n_wells):
                well_code = f'Well_{i:04d}'
                n_records = records_per_well[i]
                
                # Base characteristics for each well
                base_lon = np.random.uniform(-75, -65)
                base_lat = np.random.uniform(-35, -25)
                base_elevation = np.random.normal(500, 200)
                
                for j in range(n_records):
                    # Create synthetic dates for temporal ordering
                    date_value = pd.Timestamp('2020-01-01') + pd.Timedelta(days=j*30 + np.random.randint(0, 30))
                    
                    well_data.append({
                        'Code': well_code,
                        'Date_String': date_value,
                        'Depth to water (m)': np.random.exponential(10),
                        'Longitude_GCS_WGS_1984': base_lon + np.random.normal(0, 0.01),
                        'Latitude_GCS_WGS_1984': base_lat + np.random.normal(0, 0.01),
                        'elevation_NASADEM': base_elevation + np.random.normal(0, 50),
                        'slope_NASADEM': np.random.exponential(5),
                        'terraclim_pr_value': np.random.exponential(100),
                        'terraclim_tmmn_value': np.random.normal(100, 50),
                        'terraclim_tmmx_value': np.random.normal(200, 50),
                        'Basin': np.random.choice(['Basin_A', 'Basin_B', 'Basin_C']),
                        'Status': np.random.choice(['Active', 'Inactive', np.nan], p=[0.3, 0.3, 0.4])
                    })
            
            self.df = pd.DataFrame(well_data)
        
        self.log_time("Data Loading")
        
        # Filter data to only include rows where Status is blank/NaN
        print(f"Original dataset shape: {self.df.shape}")
        self.df_filtered = self.df[self.df['Status'].isnull()].copy()
        print(f"Filtered dataset shape (Status is blank): {self.df_filtered.shape}")
        
        if len(self.df_filtered) == 0:
            print("❌ No rows with blank Status found! Using all data for demonstration.")
            self.df_filtered = self.df.copy()
        
        # Remove rows where target variable is missing
        initial_rows = len(self.df_filtered)
        self.df_filtered = self.df_filtered.dropna(subset=[self.target_col])
        print(f"Removed {initial_rows - len(self.df_filtered)} rows with missing target values")
        
        # Prepare features
        self._prepare_features()
        self.log_time("Data Preprocessing")
        
    def _prepare_features(self):
        """Prepare feature matrix and target variable."""
        # Identify numeric and categorical columns
        numeric_cols = self.df_filtered.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = self.df_filtered.select_dtypes(include=['object']).columns.tolist()
        
        # Remove target and irrelevant columns
        if self.target_col in numeric_cols:
            numeric_cols.remove(self.target_col)
        
        columns_to_remove = ['Status', 'Code', 'Date_String']  # Keep Code for splitting but don't use as feature
        for col in columns_to_remove:
            if col in categorical_cols:
                categorical_cols.remove(col)
            if col in numeric_cols:
                numeric_cols.remove(col)
        
        # Handle missing values in numeric columns
        for col in numeric_cols:
            if self.df_filtered[col].isnull().sum() > 0:
                self.df_filtered[col].fillna(self.df_filtered[col].median(), inplace=True)
        
        # Encode categorical variables
        for col in categorical_cols:
            if col in self.df_filtered.columns:
                if self.df_filtered[col].isnull().sum() > 0:
                    self.df_filtered[col].fillna(self.df_filtered[col].mode()[0], inplace=True)
                
                le = LabelEncoder()
                self.df_filtered[col + '_encoded'] = le.fit_transform(self.df_filtered[col].astype(str))
        
        # Create feature columns list
        encoded_categorical_cols = [col + '_encoded' for col in categorical_cols if col in self.df_filtered.columns]
        self.feature_cols = numeric_cols + encoded_categorical_cols
        print(f"Final feature set: {len(self.feature_cols)} features")
        
    def create_improved_temporal_well_split(self, X, y, train_ratio=0.7, min_records_per_well=9):
        """
        Improved temporal split strategy for groundwater level prediction.
        
        This approach:
        1. Sorts data by date within each well
        2. Uses earliest records for training, latest for testing
        3. Ensures minimum records per well for reliable splitting
        4. Handles wells with insufficient data appropriately
        
        Parameters:
        - train_ratio: Proportion of records per well to use for training
        - min_records_per_well: Minimum records required per well for splitting
        """
        
        if 'Code' not in self.df_filtered.columns:
            print("Warning: 'Code' column not found. Using random split instead.")
            return train_test_split(X, y, test_size=0.3, random_state=42)
        
        # Create a date column if it doesn't exist
        if 'Date_String' not in self.df_filtered.columns:
            # If no date column exists, create one based on row order within each well
            print("Creating synthetic date ordering based on data sequence...")
            self.df_filtered['Date_String'] = self.df_filtered.groupby('Code').cumcount()
        
        train_indices = []
        test_indices = []
        wells_with_insufficient_data = []
        
        # Get unique wells
        unique_wells = self.df_filtered['Code'].unique()
        
        for well in unique_wells:
            # Get data for this well
            well_mask = self.df_filtered['Code'] == well
            well_data = self.df_filtered[well_mask].copy()
            
            # Sort by date to ensure temporal order (oldest first)
            well_data_sorted = well_data.sort_values('Date_String')
            well_indices = well_data_sorted.index.tolist()
            
            n_records = len(well_indices)
            
            # Handle wells with insufficient data
            if n_records < min_records_per_well:
                wells_with_insufficient_data.append(well)
                # For wells with few records, use them for training only
                train_indices.extend(well_indices)
                continue
            
            # Calculate split point - ensure at least 1 record in each set
            n_train = max(1, int(n_records * train_ratio))
            n_train = min(n_train, n_records - 1)  # Ensure at least 1 test record
            
            # Split indices: first n_train records (oldest) for training, rest for testing
            train_indices.extend(well_indices[:n_train])
            test_indices.extend(well_indices[n_train:])
        
        print(f"Wells with insufficient data (< {min_records_per_well} records): {len(wells_with_insufficient_data)}")
        print(f"Training records: {len(train_indices)}")
        print(f"Testing records: {len(test_indices)}")
        
        # Convert to boolean masks
        train_mask = self.df_filtered.index.isin(train_indices)
        test_mask = self.df_filtered.index.isin(test_indices)
        
        return X[train_mask], X[test_mask], y[train_mask], y[test_mask]
    
    def validate_temporal_split_quality(self, train_indices, test_indices):
        """
        Validate the quality of temporal split by checking:
        1. No data leakage (test dates should be after train dates)
        2. Balanced representation across wells
        3. Temporal coverage
        """
        
        print("\n" + "="*50)
        print("TEMPORAL SPLIT VALIDATION")
        print("="*50)
        
        # Check for data leakage if dates are available
        if 'Date_String' in self.df_filtered.columns:
            train_data = self.df_filtered.loc[train_indices]
            test_data = self.df_filtered.loc[test_indices]
            
            # Check temporal ordering within wells
            leakage_wells = []
            for well in train_data['Code'].unique():
                if well in test_data['Code'].unique():
                    well_train_dates = train_data[train_data['Code'] == well]['Date_String'].max()
                    well_test_dates = test_data[test_data['Code'] == well]['Date_String'].min()
                    if well_train_dates >= well_test_dates:
                        leakage_wells.append(well)
            
            if leakage_wells:
                print(f"⚠️  WARNING: Potential data leakage detected in {len(leakage_wells)} wells")
            else:
                print("✅ No temporal data leakage detected")
        
        # Check well distribution
        train_data = self.df_filtered.loc[train_indices]
        test_data = self.df_filtered.loc[test_indices]
        
        train_wells = len(train_data['Code'].unique())
        test_wells = len(test_data['Code'].unique())
        
        print(f"Training wells: {train_wells}")
        print(f"Testing wells: {test_wells}")
        print(f"Training/Testing ratio: {len(train_indices)}/{len(test_indices)} = {len(train_indices)/len(test_indices):.2f}")
        
        return True
        
    def run_analysis(self):
        """Run complete analysis with improved temporal well split."""
        X = self.df_filtered[self.feature_cols].values
        y = self.df_filtered[self.target_col].values
        
        print(f"\n{'='*60}")
        print(f"🔄 Analysis with Improved Temporal Well Split")
        print(f"{'='*60}")
        
        # Apply improved temporal split
        X_train, X_test, y_train, y_test = self.create_improved_temporal_well_split(
            X, y, train_ratio=0.7, min_records_per_well=9
        )
        
        print(f"Training samples: {len(X_train)} ({len(X_train)/(len(X_train)+len(X_test))*100:.1f}%)")
        print(f"Testing samples: {len(X_test)} ({len(X_test)/(len(X_train)+len(X_test))*100:.1f}%)")
        
        # Train and evaluate models
        results = self._train_and_evaluate_models(X_train, X_test, y_train, y_test)
        
        # Create visualizations
        self._create_visualizations(results, y_test)
        
        self.log_time("Analysis Complete")
        
        # Print timing summary
        self.print_timing_summary()
        
        print(f"\n🎉 Analysis completed! Results saved in '{self.output_dir}' directory")
        
    def _train_and_evaluate_models(self, X_train, X_test, y_train, y_test):
        """Train and evaluate all models."""
        results = {}
        
        for name, model in self.models.items():
            model_start_time = time.time()
            print(f"Training {name}...")
            
            # Train model
            model.fit(X_train, y_train)
            
            # Make predictions
            y_pred_train = model.predict(X_train)
            y_pred_test = model.predict(X_test)
            
            # Calculate metrics
            train_metrics = self._calculate_metrics(y_train, y_pred_train)
            test_metrics = self._calculate_metrics(y_test, y_pred_test)
            
            # Cross-validation
            cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
            cv_rmse = np.sqrt(-cv_scores.mean())
            
            model_time = (time.time() - model_start_time) / 60
            print(f"  ✅ {name} completed in {model_time:.2f} minutes")
            
            results[name] = {
                'model': model,
                'train_metrics': train_metrics,
                'test_metrics': test_metrics,
                'cv_rmse': cv_rmse,
                'predictions': y_pred_test,
                'training_time_minutes': model_time
            }
        
        return results
    
    def _calculate_metrics(self, y_true, y_pred):
        """Calculate performance metrics."""
        return {
            'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
            'mae': mean_absolute_error(y_true, y_pred),
            'r2': r2_score(y_true, y_pred),
            'mape': self.mean_absolute_percentage_error(y_true, y_pred)
        }
    
    def _create_visualizations(self, results, y_test):
        """Create comprehensive visualizations."""
        # Model performance comparison
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle('Model Performance Analysis - Improved Temporal Well Split', fontsize=16, fontweight='bold')
        
        model_names = list(results.keys())
        
        # Test MAPE comparison
        test_mape = [results[name]['test_metrics']['mape'] for name in model_names]
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
        
        bars1 = axes[0, 0].bar(model_names, test_mape, color=colors)
        axes[0, 0].set_ylabel('Test MAPE (%)')
        axes[0, 0].set_title('Model Performance (Test MAPE)')
        axes[0, 0].tick_params(axis='x', rotation=45)
        
        # Add value labels
        for bar, value in zip(bars1, test_mape):
            axes[0, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(test_mape)*0.01,
                           f'{value:.2f}%', ha='center', va='bottom', fontweight='bold')
        
        # Test R² comparison
        test_r2 = [results[name]['test_metrics']['r2'] for name in model_names]
        bars2 = axes[0, 1].bar(model_names, test_r2, color=colors)
        axes[0, 1].set_ylabel('Test R² Score')
        axes[0, 1].set_title('Model Performance (Test R²)')
        axes[0, 1].tick_params(axis='x', rotation=45)
        
        for bar, value in zip(bars2, test_r2):
            axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(test_r2)*0.01,
                           f'{value:.3f}', ha='center', va='bottom', fontweight='bold')
        
        # Training vs Testing MAPE
        train_mape = [results[name]['train_metrics']['mape'] for name in model_names]
        x_pos = np.arange(len(model_names))
        width = 0.35
        
        axes[0, 2].bar(x_pos - width/2, train_mape, width, label='Train MAPE', color='lightblue', alpha=0.8)
        axes[0, 2].bar(x_pos + width/2, test_mape, width, label='Test MAPE', color='lightcoral', alpha=0.8)
        axes[0, 2].set_ylabel('MAPE (%)')
        axes[0, 2].set_title('Training vs Testing MAPE')
        axes[0, 2].set_xticks(x_pos)
        axes[0, 2].set_xticklabels(model_names, rotation=45)
        axes[0, 2].legend()
        
        # Predicted vs Actual plots for each model
        for i, (name, color) in enumerate(zip(model_names, colors)):
            y_pred = results[name]['predictions']
            ax = axes[1, i]
            
            ax.scatter(y_test, y_pred, alpha=0.6, color=color, s=20, edgecolors='white', linewidth=0.5)
            ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
            ax.set_xlabel('Actual Depth to Water (m)')
            ax.set_ylabel('Predicted Depth to Water (m)')
            ax.set_title(f'{name}: Predicted vs Actual')
            
            # Add metrics and timing
            r2 = results[name]['test_metrics']['r2']
            mape = results[name]['test_metrics']['mape']
            train_time = results[name]['training_time_minutes']
            ax.text(0.05, 0.95, f'R² = {r2:.3f}\nMAPE = {mape:.2f}%\nTime = {train_time:.2f}m', 
                   transform=ax.transAxes, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        plt.tight_layout()
        plt.savefig(self.output_dir / 'temporal_model_performance.png', 
                   bbox_inches='tight', facecolor='white')
        plt.show()
        
        # Feature importance analysis
        self._create_feature_importance_plot(results)
        
        # Residuals analysis
        self._create_residuals_analysis(results, y_test)
        
        # Performance summary
        self._create_performance_summary(results)
    
    def _create_feature_importance_plot(self, results):
        """Create feature importance visualization."""
        fig, axes = plt.subplots(1, 3, figsize=(20, 6))
        fig.suptitle('Feature Importance Analysis - Temporal Well Split', fontsize=16, fontweight='bold')
        
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
        
        for i, (name, color) in enumerate(zip(results.keys(), colors)):
            model = results[name]['model']
            
            if hasattr(model, 'feature_importances_'):
                importance_scores = model.feature_importances_
                
                # Get top 15 features
                indices = np.argsort(importance_scores)[::-1][:15]
                top_features = [self.feature_cols[idx] for idx in indices]
                top_scores = importance_scores[indices]
                
                # Truncate long feature names
                display_names = [f[:25] + '...' if len(f) > 25 else f for f in top_features]
                
                axes[i].barh(range(len(top_features)), top_scores, color=color, alpha=0.8)
                axes[i].set_yticks(range(len(top_features)))
                axes[i].set_yticklabels(display_names)
                axes[i].set_xlabel('Importance Score')
                axes[i].set_title(f'{name} - Top 15 Features')
                axes[i].invert_yaxis()
                
                # Add value labels
                for j, score in enumerate(top_scores):
                    axes[i].text(score + max(top_scores)*0.01, j, f'{score:.3f}', 
                               va='center', fontsize=8)
        
        plt.tight_layout()
        plt.savefig(self.output_dir / 'temporal_feature_importance.png', 
                   bbox_inches='tight', facecolor='white')
        plt.show()
    
    def _create_residuals_analysis(self, results, y_test):
        """Create residuals analysis plots."""
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle('Residuals Analysis - Temporal Well Split', fontsize=16, fontweight='bold')
        
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
        
        for i, (name, color) in enumerate(zip(results.keys(), colors)):
            y_pred = results[name]['predictions']
            residuals = y_test - y_pred
            
            # Residuals vs Predicted
            axes[0, i].scatter(y_pred, residuals, alpha=0.6, color=color, s=20)
            axes[0, i].axhline(y=0, color='red', linestyle='--', linewidth=2)
            axes[0, i].set_xlabel('Predicted Values')
            axes[0, i].set_ylabel('Residuals')
            axes[0, i].set_title(f'{name}: Residuals vs Predicted')
            
            # Residuals histogram
            axes[1, i].hist(residuals, bins=30, alpha=0.7, color=color, edgecolor='black')
            axes[1, i].set_xlabel('Residuals')
            axes[1, i].set_ylabel('Frequency')
            axes[1, i].set_title(f'{name}: Residuals Distribution')
            axes[1, i].axvline(residuals.mean(), color='red', linestyle='--', 
                              label=f'Mean: {residuals.mean():.2f}')
            axes[1, i].legend()
        
        plt.tight_layout()
        plt.savefig(self.output_dir / 'temporal_residuals_analysis.png', 
                   bbox_inches='tight', facecolor='white')
        plt.show()
    
    def _create_performance_summary(self, results):
        """Create performance summary table."""
        fig, ax = plt.subplots(figsize=(12, 8))
        fig.suptitle('Performance Summary - Improved Temporal Well Split', fontsize=16, fontweight='bold')
        
        # Create summary data
        summary_data = []
        for model_name in results.keys():
            train_metrics = results[model_name]['train_metrics']
            test_metrics = results[model_name]['test_metrics']
            cv_rmse = results[model_name]['cv_rmse']
            train_time = results[model_name]['training_time_minutes']
            
            summary_data.append([
                model_name,
                f"{train_metrics['mape']:.2f}%",
                f"{test_metrics['mape']:.2f}%",
                f"{train_metrics['r2']:.3f}",
                f"{test_metrics['r2']:.3f}",
                f"{cv_rmse:.2f}",
                f"{train_time:.2f}m"
            ])
        
        table = ax.table(cellText=summary_data,
                        colLabels=['Model', 'Train MAPE', 'Test MAPE', 'Train R²', 'Test R²', 'CV RMSE', 'Time'],
                        cellLoc='center',
                        loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1.2, 2)
        
        # Color code the headers
        for i in range(len(summary_data[0])):
            table[(0, i)].set_facecolor('#4ECDC4')
            table[(0, i)].set_text_props(weight='bold', color='white')
        
        ax.axis('off')
        ax.set_title('Model Performance Comparison\n(Realistic Temporal Split for Groundwater Prediction)', 
                    pad=20, fontsize=14)
        
        plt.tight_layout()
        plt.savefig(self.output_dir / 'temporal_performance_summary.png', 
                   bbox_inches='tight', facecolor='white')
        plt.show()
        
        # Print summary to console
        print("\n" + "="*80)
        print("📊 PERFORMANCE SUMMARY - IMPROVED TEMPORAL WELL SPLIT")
        print("="*80)
        print("This approach uses historical data to predict future groundwater levels,")
        print("providing the most realistic assessment for operational use.")
        print("-"*80)
        
        for model_name in results.keys():
            test_metrics = results[model_name]['test_metrics']
            train_time = results[model_name]['training_time_minutes']
            print(f"\n{model_name}:")
            print(f"  Test MAPE: {test_metrics['mape']:.2f}%")
            print(f"  Test R²: {test_metrics['r2']:.3f}")
            print(f"  Test RMSE: {test_metrics['rmse']:.2f}")
            print(f"  Test MAE: {test_metrics['mae']:.2f}")
            print(f"  Training Time: {train_time:.2f} minutes")
        
        print("\n" + "="*80)
        print("🔍 ANALYSIS INSIGHTS:")
        print("• Temporal split provides realistic performance estimates for time-series prediction")
        print("• Training on historical data and testing on future data mimics real-world deployment")
        print("• Feature importance reveals which variables are most predictive of groundwater levels")
        print("• Cross-validation scores help assess model stability and generalization")
        print("• MAPE provides intuitive percentage error interpretation for stakeholders")
        print("• R² score indicates how well the model explains groundwater level variance")
        print("="*80)
        
        # Find best performing model
        best_model_name = min(results.keys(), key=lambda x: results[x]['test_metrics']['mape'])
        best_mape = results[best_model_name]['test_metrics']['mape']
        best_r2 = results[best_model_name]['test_metrics']['r2']
        
        print(f"\n🏆 BEST PERFORMING MODEL: {best_model_name}")
        print(f"   Test MAPE: {best_mape:.2f}%")
        print(f"   Test R²: {best_r2:.3f}")
        print(f"   This model provides the most accurate predictions for operational use.")
        print("="*80)
        
        # Data quality and splitting insights
        print("\n📋 DATA SPLITTING INSIGHTS:")
        print("• Wells with multiple time-series records enable proper temporal validation")
        print("• Early measurements used for training, recent measurements for testing")
        print("• This approach prevents data leakage and overly optimistic performance estimates")
        print("• Results are more representative of real-world model deployment scenarios")
        print("="*80)
        
        # Save summary to text file
        summary_file = self.output_dir / 'analysis_summary.txt'
        with open(summary_file, 'w') as f:
            f.write("GROUNDWATER DEPTH PREDICTION ANALYSIS SUMMARY\n")
            f.write("=" * 50 + "\n\n")
            f.write("Analysis Method: Improved Temporal Well Split\n")
            f.write(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
            
            f.write("MODEL PERFORMANCE RESULTS:\n")
            f.write("-" * 30 + "\n")
            for model_name in results.keys():
                test_metrics = results[model_name]['test_metrics']
                train_time = results[model_name]['training_time_minutes']
                f.write(f"\n{model_name}:\n")
                f.write(f"  Test MAPE: {test_metrics['mape']:.2f}%\n")
                f.write(f"  Test R²: {test_metrics['r2']:.3f}\n")
                f.write(f"  Test RMSE: {test_metrics['rmse']:.2f}\n")
                f.write(f"  Test MAE: {test_metrics['mae']:.2f}\n")
                f.write(f"  Training Time: {train_time:.2f} minutes\n")
            
            f.write(f"\nBest Model: {best_model_name} (MAPE: {best_mape:.2f}%)\n")
            f.write(f"Dataset Shape: {self.df_filtered.shape}\n")
            f.write(f"Number of Features: {len(self.feature_cols)}\n")
        
        print(f"\n💾 Analysis summary saved to: {summary_file}")
        print("="*80)

# Usage
def main():
    # Update this path to your actual file location
    file_path = r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\results\chile_2025\groundwater_chile_and_elevation_dataset_2025_with_GEE_7_2_25.xlsx"
    
    # Initialize and run analysis
    predictor = GroundwaterPredictor(file_path)
    predictor.load_and_preprocess_data()
    predictor.run_analysis()

if __name__ == "__main__":
    main()

#### Testing the minimum amount

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance
import warnings
import time
from pathlib import Path
import os
from datetime import datetime
warnings.filterwarnings('ignore')

# Set professional plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 9

class GroundwaterPredictor:
    """Professional groundwater depth prediction analysis with improved temporal well split."""
    
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None
        self.df_filtered = None
        self.results = {}
        self.predictions = {}
        self.feature_cols = []
        self.target_col = 'Depth to water (m)'
        self.models = {
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            'Decision Tree': DecisionTreeRegressor(random_state=42)
        }
        
        # Timing variables
        self.start_time = None
        self.timing_log = []
        
        # Create output directory for figures
        self.output_dir = Path("groundwater_temporal_analysis_results")
        self.output_dir.mkdir(exist_ok=True)
        
        # To store results from sensitivity analysis
        self.sensitivity_results = {}
        
    def log_time(self, step_name):
        """Log timing information for each step."""
        current_time = time.time()
        if self.start_time is None:
            self.start_time = current_time
            elapsed = 0
        else:
            elapsed = current_time - self.start_time
        
        self.timing_log.append({
            'step': step_name,
            'elapsed_minutes': elapsed / 60,
            'timestamp': datetime.now().strftime('%H:%M:%S')
        })
        
        print(f"⏱️  {step_name} completed at {datetime.now().strftime('%H:%M:%S')} "
              f"(Total elapsed: {elapsed/60:.2f} minutes)")
        
    def print_timing_summary(self):
        """Print a summary of timing information."""
        if not self.timing_log:
            return
            
        print("\n" + "="*60)
        print("⏰ TIMING SUMMARY")
        print("="*60)
        
        for i, entry in enumerate(self.timing_log):
            if i == 0:
                step_time = entry['elapsed_minutes']
            else:
                step_time = entry['elapsed_minutes'] - self.timing_log[i-1]['elapsed_minutes']
            
            print(f"{entry['step']:<40} | {step_time:>8.2f} min | {entry['timestamp']}")
        
        total_time = self.timing_log[-1]['elapsed_minutes']
        print("-" * 60)
        print(f"{'TOTAL RUNTIME':<40} | {total_time:>8.2f} min")
        print("="*60)
    
    def mean_absolute_percentage_error(self, y_true, y_pred):
        """Calculate Mean Absolute Percentage Error (MAPE)"""
        epsilon = 1e-10
        return np.mean(np.abs((y_true - y_pred) / (y_true + epsilon))) * 100
    
    def load_and_preprocess_data(self):
        """Load and preprocess the dataset."""
        print("🌊 Chile Groundwater Depth Prediction Analysis")
        print("📊 Using Improved Temporal Well Split Strategy")
        print("=" * 60)
        print(f"🚀 Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("\n📊 Loading and preprocessing dataset...")
        
        self.log_time("Analysis Started")
        
        try:
            if self.file_path.endswith('.csv'):
                self.df = pd.read_csv(self.file_path)
            else:
                self.df = pd.read_excel(self.file_path)
            print(f"✅ Dataset loaded successfully! Shape: {self.df.shape}")
        except Exception as e:
            print(f"❌ Error loading file: {e}")
            # Create sample data for demonstration
            print("Creating sample data for demonstration...")
            np.random.seed(42)
            n_samples = 10000
            n_wells = 1000
            records_per_well = np.random.poisson(15, n_wells) + 1
            well_data = []
            for i in range(n_wells):
                well_code = f'Well_{i:04d}'
                n_records = records_per_well[i]
                base_lon = np.random.uniform(-75, -65); base_lat = np.random.uniform(-35, -25); base_elevation = np.random.normal(500, 200)
                for j in range(n_records):
                    date_value = pd.Timestamp('2010-01-01') + pd.Timedelta(days=j*90 + np.random.randint(0, 90))
                    well_data.append({
                        'Code': well_code, 'Date_String': date_value, 'Depth to water (m)': np.random.exponential(10) + j*0.1,
                        'Longitude_GCS_WGS_1984': base_lon + np.random.normal(0, 0.01), 'Latitude_GCS_WGS_1984': base_lat + np.random.normal(0, 0.01),
                        'elevation_NASADEM': base_elevation + np.random.normal(0, 50), 'slope_NASADEM': np.random.exponential(5),
                        'terraclim_pr_value': np.random.exponential(100), 'terraclim_tmmn_value': np.random.normal(100, 50),
                        'terraclim_tmmx_value': np.random.normal(200, 50), 'Basin': np.random.choice(['Basin_A', 'Basin_B', 'Basin_C']),
                        'Status': np.random.choice(['Active', 'Inactive', np.nan], p=[0.3, 0.3, 0.4])
                    })
            self.df = pd.DataFrame(well_data)

        self.log_time("Data Loading")
        
        print(f"Original dataset shape: {self.df.shape}")
        self.df_filtered = self.df[self.df['Status'].isnull()].copy()
        print(f"Filtered dataset shape (Status is blank): {self.df_filtered.shape}")
        
        if len(self.df_filtered) == 0:
            print("❌ No rows with blank Status found! Using all data for demonstration.")
            self.df_filtered = self.df.copy()
        
        initial_rows = len(self.df_filtered)
        self.df_filtered = self.df_filtered.dropna(subset=[self.target_col])
        print(f"Removed {initial_rows - len(self.df_filtered)} rows with missing target values")
        
        self._prepare_features()
        self.log_time("Data Preprocessing")
        
    def _prepare_features(self):
        """Prepare feature matrix and target variable."""
        numeric_cols = self.df_filtered.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = self.df_filtered.select_dtypes(include=['object']).columns.tolist()
        
        if self.target_col in numeric_cols:
            numeric_cols.remove(self.target_col)
        
        columns_to_remove = ['Status', 'Code', 'Date_String']
        for col in columns_to_remove:
            if col in categorical_cols: categorical_cols.remove(col)
            if col in numeric_cols: numeric_cols.remove(col)
        
        for col in numeric_cols:
            if self.df_filtered[col].isnull().sum() > 0: self.df_filtered[col].fillna(self.df_filtered[col].median(), inplace=True)
        
        for col in categorical_cols:
            if col in self.df_filtered.columns:
                if self.df_filtered[col].isnull().sum() > 0: self.df_filtered[col].fillna(self.df_filtered[col].mode()[0], inplace=True)
                le = LabelEncoder()
                self.df_filtered[col + '_encoded'] = le.fit_transform(self.df_filtered[col].astype(str))
        
        encoded_categorical_cols = [col + '_encoded' for col in categorical_cols if col in self.df_filtered.columns]
        self.feature_cols = numeric_cols + encoded_categorical_cols
        print(f"Final feature set: {len(self.feature_cols)} features")
        
    def create_improved_temporal_well_split(self, X, y, train_ratio=0.7, min_records_per_well=3):
        """Improved temporal split ensuring minimum records per well."""
        if 'Code' not in self.df_filtered.columns:
            print("Warning: 'Code' column not found. Using random split instead.")
            return train_test_split(X, y, test_size=0.3, random_state=42)
        
        if 'Date_String' not in self.df_filtered.columns:
            print("Creating synthetic date ordering based on data sequence...")
            self.df_filtered['Date_String'] = self.df_filtered.groupby('Code').cumcount()
        
        train_indices, test_indices, wells_with_insufficient_data = [], [], []
        unique_wells = self.df_filtered['Code'].unique()
        
        for well in unique_wells:
            well_mask = self.df_filtered['Code'] == well
            well_data_sorted = self.df_filtered[well_mask].copy().sort_values('Date_String')
            well_indices = well_data_sorted.index.tolist()
            n_records = len(well_indices)
            
            if n_records < min_records_per_well:
                wells_with_insufficient_data.append(well)
                train_indices.extend(well_indices)
                continue
            
            n_train = max(1, int(n_records * train_ratio))
            n_train = min(n_train, n_records - 1)
            train_indices.extend(well_indices[:n_train])
            test_indices.extend(well_indices[n_train:])
        
        print(f"Wells with insufficient data (< {min_records_per_well} records): {len(wells_with_insufficient_data)} (used for training)")
        print(f"Training records: {len(train_indices)}")
        print(f"Testing records: {len(test_indices)}")
        
        train_mask = self.df_filtered.index.isin(train_indices)
        test_mask = self.df_filtered.index.isin(test_indices)
        return X[train_mask], X[test_mask], y[train_mask], y[test_mask]
        
    def run_analysis(self, min_records_per_well):
        """Run complete analysis for a given min_records_per_well, returning results."""
        X = self.df_filtered[self.feature_cols].values
        y = self.df_filtered[self.target_col].values
        
        print(f"\n{'='*60}")
        print(f"🔄 Running Analysis for min_records_per_well = {min_records_per_well}")
        print(f"{'='*60}")
        
        X_train, X_test, y_train, y_test = self.create_improved_temporal_well_split(
            X, y, train_ratio=0.7, min_records_per_well=min_records_per_well
        )
        
        if len(X_test) == 0:
            print(f"⚠️  WARNING: No testing data available for min_records_per_well = {min_records_per_well}. Skipping this run.")
            return ({}, len(X_train), len(X_test))

        print(f"Training samples: {len(X_train)} ({len(X_train)/(len(X_train)+len(X_test))*100:.1f}%)")
        print(f"Testing samples: {len(X_test)} ({len(X_test)/(len(X_train)+len(X_test))*100:.1f}%)")
        
        results = self._train_and_evaluate_models(X_train, X_test, y_train, y_test)
        
        run_id = f"min_records_{min_records_per_well}"
        self._create_visualizations(results, y_train, y_test, run_id)
        
        print(f"\n🎉 Analysis for {run_id} completed! Individual results saved in '{self.output_dir}' directory.")
        return results, len(X_train), len(X_test)

    def run_sensitivity_analysis(self, min_records_values):
        """Run the analysis for a range of min_records_per_well values and plot summary."""
        print("\n" + "#"*80)
        print("🔬 SENSITIVITY ANALYSIS: Impact of 'min_records_per_well'")
        print(f"🔬 Running for values: {min_records_values}")
        print("#"*80)

        for min_records in min_records_values:
            self.log_time(f"Starting run for min_records = {min_records}")
            results, n_train, n_test = self.run_analysis(min_records_per_well=min_records)
            
            if not results: continue # Skip if run was aborted

            self.sensitivity_results[min_records] = {'results': results, 'n_train': n_train, 'n_test': n_test}
            self.log_time(f"Completed run for min_records = {min_records}")

        self.plot_sensitivity_summary()
        self.print_timing_summary()
        print(f"\n🎉 Sensitivity analysis completed! All results saved in '{self.output_dir}' directory")

    def plot_sensitivity_summary(self):
        """Plots the summary of the sensitivity analysis and prints a summary table."""
        print("\n" + "="*60); print("📈 Plotting Sensitivity Analysis Summary"); print("="*60)
        if not self.sensitivity_results:
            print("No sensitivity results to plot.")
            return

        min_records_values = sorted(self.sensitivity_results.keys())
        model_names = list(self.models.keys())
        metrics_to_plot = {'Test R²': 'r2', 'Test MAPE (%)': 'mape', 'Test RMSE': 'rmse'}
        plot_data = {metric: {model: [] for model in model_names} for metric in metrics_to_plot}
        sample_counts = {'train': [], 'test': []}
        
        for val in min_records_values:
            run_data = self.sensitivity_results[val]
            sample_counts['train'].append(run_data['n_train'])
            sample_counts['test'].append(run_data['n_test'])
            for model in model_names:
                for metric_name, metric_key in metrics_to_plot.items():
                    score = run_data['results'][model]['test_metrics'][metric_key]
                    plot_data[metric_name][model].append(score)
        
        # --- START: New code to create and print the summary table ---
        table_rows = []
        # Create header with short model names for compactness
        header = ['min_records', 'Train Samples', 'Test Samples']
        for model in model_names:
            model_short_name = ''.join(word[0] for word in model.split())
            header.extend([f'{model_short_name} Test R²', f'{model_short_name} Test MAPE', f'{model_short_name} Test RMSE'])
        
        # Create data rows for the table
        for i, val in enumerate(min_records_values):
            row = [
                val,
                sample_counts['train'][i],
                sample_counts['test'][i]
            ]
            for model in model_names:
                row.extend([
                    plot_data['Test R²'][model][i],
                    plot_data['Test MAPE (%)'][model][i],
                    plot_data['Test RMSE'][model][i]
                ])
            table_rows.append(row)
            
        # Use pandas to create and display a nicely formatted table
        summary_df = pd.DataFrame(table_rows, columns=header)
        summary_df = summary_df.set_index('min_records')
        
        # Define specific formatting for each column type for better readability
        formatters = {
            'Train Samples': '{:,}'.format,
            'Test Samples': '{:,}'.format,
        }
        for col in summary_df.columns:
            if 'R²' in col:
                formatters[col] = '{:.3f}'.format
            elif 'RMSE' in col:
                formatters[col] = '{:.3f}'.format
            elif 'MAPE' in col:
                formatters[col] = '{:.2f}%'.format

        print("\n📈 Sensitivity Analysis Summary Table:")
        # Print the formatted dataframe to the console
        print(summary_df.to_string(formatters=formatters))
        print("="*len(summary_df.to_string(formatters=formatters).splitlines()[0]) + "\n")
        # --- END: New code ---

        # The original plotting code remains unchanged
        fig, axes = plt.subplots(2, 2, figsize=(18, 14), dpi=300)
        fig.suptitle("Sensitivity Analysis: Impact of 'min_records_per_well' on Model Performance", fontsize=18, fontweight='bold')
        
        ax1 = axes[0, 0]
        ax1.plot(min_records_values, sample_counts['train'], 'o-', label='Training Samples')
        ax1.plot(min_records_values, sample_counts['test'], 's-', label='Testing Samples')
        ax1.set_xlabel("'min_records_per_well' (log scale)"); ax1.set_ylabel("Number of Samples")
        ax1.set_title("Effect on Dataset Size"); ax1.set_xticks(min_records_values)
        ax1.get_xaxis().set_major_formatter(plt.ScalarFormatter()); ax1.set_xscale('log')
        ax1.legend(); ax1.grid(True, which="both", ls="--")
        
        metric_axes = [axes[0, 1], axes[1, 0], axes[1, 1]]
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
        
        for ax, metric_name in zip(metric_axes, metrics_to_plot.keys()):
            for model, color in zip(model_names, colors):
                ax.plot(min_records_values, plot_data[metric_name][model], 'o-', label=model, color=color)
            ax.set_xlabel("'min_records_per_well' (log scale)"); ax.set_ylabel(metric_name)
            ax.set_title(f"{metric_name} vs. 'min_records_per_well'"); ax.set_xticks(min_records_values)
            ax.get_xaxis().set_major_formatter(plt.ScalarFormatter()); ax.set_xscale('log')
            ax.legend(); ax.grid(True, which="both", ls="--")

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.savefig(self.output_dir / 'sensitivity_analysis_summary.png', bbox_inches='tight', facecolor='white')
        plt.show()
    
    def _train_and_evaluate_models(self, X_train, X_test, y_train, y_test):
        """Train and evaluate all models."""
        results = {}
        for name, model in self.models.items():
            model_start_time = time.time()
            print(f"Training {name}...")
            model.fit(X_train, y_train)
            y_pred_train, y_pred_test = model.predict(X_train), model.predict(X_test)
            train_metrics, test_metrics = self._calculate_metrics(y_train, y_pred_train), self._calculate_metrics(y_test, y_pred_test)
            cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
            model_time = (time.time() - model_start_time) / 60
            print(f"  ✅ {name} completed in {model_time:.2f} minutes")
            results[name] = {
                'model': model, 'train_metrics': train_metrics, 'test_metrics': test_metrics,
                'cv_rmse': np.sqrt(-cv_scores.mean()), 'train_predictions': y_pred_train, 'test_predictions': y_pred_test, 'training_time_minutes': model_time
            }
        return results
    
    def _calculate_metrics(self, y_true, y_pred):
        """Calculate performance metrics."""
        return {
            'rmse': np.sqrt(mean_squared_error(y_true, y_pred)), 'mae': mean_absolute_error(y_true, y_pred),
            'r2': r2_score(y_true, y_pred), 'mape': self.mean_absolute_percentage_error(y_true, y_pred)
        }
    
    def _create_visualizations(self, results, y_train, y_test, run_id):
        """Create comprehensive visualizations for a specific run."""
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        min_records_val = run_id.split('_')[-1]
        title = f'Model Performance (min_records_per_well = {min_records_val})'
        fig.suptitle(title, fontsize=16, fontweight='bold')
        
        model_names = list(results.keys())
        test_mape = [results[name]['test_metrics']['mape'] for name in model_names]
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
        
        bars1 = axes[0, 0].bar(model_names, test_mape, color=colors)
        axes[0, 0].set_ylabel('Test MAPE (%)'); axes[0, 0].set_title('Model Performance (Test MAPE)'); axes[0, 0].tick_params(axis='x', rotation=45)
        for bar, value in zip(bars1, test_mape): axes[0, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(test_mape)*0.01, f'{value:.2f}%', ha='center', va='bottom', fontweight='bold')
        
        test_r2 = [results[name]['test_metrics']['r2'] for name in model_names]
        bars2 = axes[0, 1].bar(model_names, test_r2, color=colors)
        axes[0, 1].set_ylabel('Test R² Score'); axes[0, 1].set_title('Model Performance (Test R²)'); axes[0, 1].tick_params(axis='x', rotation=45)
        for bar, value in zip(bars2, test_r2): axes[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(test_r2)*0.01, f'{value:.3f}', ha='center', va='bottom', fontweight='bold')
        
        train_mape = [results[name]['train_metrics']['mape'] for name in model_names]
        x_pos = np.arange(len(model_names)); width = 0.35
        axes[0, 2].bar(x_pos - width/2, train_mape, width, label='Train MAPE', color='lightblue', alpha=0.8)
        axes[0, 2].bar(x_pos + width/2, test_mape, width, label='Test MAPE', color='lightcoral', alpha=0.8)
        axes[0, 2].set_ylabel('MAPE (%)'); axes[0, 2].set_title('Training vs Testing MAPE'); axes[0, 2].set_xticks(x_pos); axes[0, 2].set_xticklabels(model_names, rotation=45); axes[0, 2].legend()
        
        # Modified 1-to-1 plots to include both training and testing data
        for i, (name, color) in enumerate(zip(model_names, colors)):
            ax = axes[1, i]
            
            # Plot training data with circles
            ax.scatter(y_train, results[name]['train_predictions'], alpha=0.4, color='lightblue', 
                      s=15, edgecolors='navy', linewidth=0.3, label=f'Training (n={len(y_train)})', marker='o')
            
            # Plot testing data with squares
            ax.scatter(y_test, results[name]['test_predictions'], alpha=0.7, color=color, 
                      s=20, edgecolors='white', linewidth=0.5, label=f'Testing (n={len(y_test)})', marker='s')
            
            # 1:1 line
            all_values = np.concatenate([y_train, y_test])
            min_val, max_val = all_values.min(), all_values.max()
            ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='1:1 Line')
            
            ax.set_xlabel('Actual Depth to Water (m)')
            ax.set_ylabel('Predicted Depth to Water (m)')
            ax.set_title(f'{name}: Predicted vs Actual')
            
            # Add metrics text box
            train_r2 = results[name]['train_metrics']['r2']
            test_r2 = results[name]['test_metrics']['r2']
            train_mape = results[name]['train_metrics']['mape']
            test_mape_val = results[name]['test_metrics']['mape']
            train_time = results[name]['training_time_minutes']
            
            metrics_text = f'Train R² = {train_r2:.3f}\nTest R² = {test_r2:.3f}\nTrain MAPE = {train_mape:.2f}%\nTest MAPE = {test_mape_val:.2f}%\nTime = {train_time:.2f}m'
            ax.text(0.05, 0.95, metrics_text, transform=ax.transAxes, 
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8), 
                   verticalalignment='top', fontsize=8)
            
            # Add legend
            ax.legend(loc='lower right', fontsize=8)
            
            # Set equal aspect ratio for true 1:1 visualization
            ax.set_aspect('equal', adjustable='box')
        
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.savefig(self.output_dir / f'model_performance_{run_id}.png', bbox_inches='tight', facecolor='white')
        plt.close(fig) # Close figure to avoid displaying it now
        
        # Create separate detailed 1-to-1 plots
        self._create_detailed_one_to_one_plots(results, y_train, y_test, run_id)
        
        self._create_feature_importance_plot(results, run_id)
        self._create_residuals_analysis(results, y_test, run_id)
        self._create_performance_summary(results, run_id)
    
    def _create_detailed_one_to_one_plots(self, results, y_train, y_test, run_id):
        """Create detailed 1-to-1 plots for each model showing training and testing data."""
        model_names = list(results.keys())
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
        
        # Create a large figure with one subplot per model
        fig, axes = plt.subplots(1, 3, figsize=(20, 6))
        min_records_val = run_id.split('_')[-1]
        fig.suptitle(f'Detailed 1-to-1 Plots (min_records_per_well = {min_records_val})', 
                     fontsize=16, fontweight='bold')
        
        for i, (name, color) in enumerate(zip(model_names, colors)):
            ax = axes[i]
            
            # Plot training data
            train_scatter = ax.scatter(y_train, results[name]['train_predictions'], 
                                     alpha=0.5, color='lightblue', s=25, 
                                     edgecolors='darkblue', linewidth=0.5, 
                                     label=f'Training (n={len(y_train):,})', marker='o')
            
            # Plot testing data
            test_scatter = ax.scatter(y_test, results[name]['test_predictions'], 
                                    alpha=0.8, color=color, s=30, 
                                    edgecolors='white', linewidth=0.8, 
                                    label=f'Testing (n={len(y_test):,})', marker='s')
            
            # 1:1 perfect prediction line
            all_values = np.concatenate([y_train, y_test])
            min_val, max_val = all_values.min(), all_values.max()
            ax.plot([min_val, max_val], [min_val, max_val], 'r--', lw=3, 
                   label='Perfect Prediction (1:1)', alpha=0.8)
            
            # Formatting
            ax.set_xlabel('Actual Depth to Water (m)', fontsize=12)
            ax.set_ylabel('Predicted Depth to Water (m)', fontsize=12)
            ax.set_title(f'{name}', fontsize=14, fontweight='bold')
            ax.grid(True, alpha=0.3)
            
            # Add comprehensive metrics
            train_metrics = results[name]['train_metrics']
            test_metrics = results[name]['test_metrics']
            
            metrics_text = (f'Training:\n'
                          f'  R² = {train_metrics["r2"]:.3f}\n'
                          f'  MAPE = {train_metrics["mape"]:.2f}%\n'
                          f'  RMSE = {train_metrics["rmse"]:.2f}\n'
                          f'Testing:\n'
                          f'  R² = {test_metrics["r2"]:.3f}\n'
                          f'  MAPE = {test_metrics["mape"]:.2f}%\n'
                          f'  RMSE = {test_metrics["rmse"]:.2f}')
            
            ax.text(0.02, 0.98, metrics_text, transform=ax.transAxes, 
                   bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.9, edgecolor='gray'), 
                   verticalalignment='top', fontsize=10, family='monospace')
            
            # Legend
            ax.legend(loc='lower right', fontsize=10, framealpha=0.9)
            
            # Set equal aspect ratio
            ax.set_aspect('equal', adjustable='box')
        
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.savefig(self.output_dir / f'detailed_one_to_one_plots_{run_id}.png', 
                   bbox_inches='tight', facecolor='white', dpi=300)
        plt.close(fig)
    
    def _create_feature_importance_plot(self, results, run_id):
        """Create feature importance visualization for a specific run."""
        fig, axes = plt.subplots(1, 3, figsize=(20, 8))
        min_records_val = run_id.split('_')[-1]
        title = f'Feature Importance (min_records_per_well = {min_records_val})'
        fig.suptitle(title, fontsize=16, fontweight='bold')
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
        
        for i, (name, color) in enumerate(zip(results.keys(), colors)):
            if hasattr(results[name]['model'], 'feature_importances_'):
                importances = results[name]['model'].feature_importances_
                indices = np.argsort(importances)[::-1][:15]
                top_features = [self.feature_cols[idx] for idx in indices]
                display_names = [f[:25] + '...' if len(f) > 25 else f for f in top_features]
                axes[i].barh(range(len(indices)), importances[indices], color=color, alpha=0.8)
                axes[i].set_yticks(range(len(indices))); axes[i].set_yticklabels(display_names)
                axes[i].set_xlabel('Importance Score'); axes[i].set_title(f'{name} - Top 15 Features'); axes[i].invert_yaxis()
                for j, score in enumerate(importances[indices]): axes[i].text(score + max(importances)*0.01, j, f'{score:.3f}', va='center', fontsize=8)

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.savefig(self.output_dir / f'feature_importance_{run_id}.png', bbox_inches='tight', facecolor='white')
        plt.close(fig)
    
    def _create_residuals_analysis(self, results, y_test, run_id):
        """Create residuals analysis plots for a specific run."""
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        min_records_val = run_id.split('_')[-1]
        title = f'Residuals Analysis (min_records_per_well = {min_records_val})'
        fig.suptitle(title, fontsize=16, fontweight='bold')
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

        for i, (name, color) in enumerate(zip(results.keys(), colors)):
            residuals = y_test - results[name]['test_predictions']
            axes[0, i].scatter(results[name]['test_predictions'], residuals, alpha=0.6, color=color, s=20)
            axes[0, i].axhline(y=0, color='red', linestyle='--', linewidth=2)
            axes[0, i].set_xlabel('Predicted Values'); axes[0, i].set_ylabel('Residuals'); axes[0, i].set_title(f'{name}: Residuals vs Predicted')
            axes[1, i].hist(residuals, bins=30, alpha=0.7, color=color, edgecolor='black')
            axes[1, i].set_xlabel('Residuals'); axes[1, i].set_ylabel('Frequency'); axes[1, i].set_title(f'{name}: Residuals Distribution')
            axes[1, i].axvline(residuals.mean(), color='red', linestyle='--', label=f'Mean: {residuals.mean():.2f}'); axes[1, i].legend()

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.savefig(self.output_dir / f'residuals_analysis_{run_id}.png', bbox_inches='tight', facecolor='white')
        plt.close(fig)
    
    def _create_performance_summary(self, results, run_id):
        """Create performance summary table and text file for a specific run."""
        fig, ax = plt.subplots(figsize=(12, 4))
        min_records_val = run_id.split('_')[-1]
        title = f'Performance Summary (min_records_per_well = {min_records_val})'
        fig.suptitle(title, fontsize=16, fontweight='bold')
        
        summary_data = []
        for name in results.keys():
            train_m, test_m = results[name]['train_metrics'], results[name]['test_metrics']
            summary_data.append([ name, f"{train_m['mape']:.2f}%", f"{test_m['mape']:.2f}%", f"{train_m['r2']:.3f}", f"{test_m['r2']:.3f}", f"{results[name]['cv_rmse']:.2f}", f"{results[name]['training_time_minutes']:.2f}m" ])
        
        table = ax.table(cellText=summary_data, colLabels=['Model', 'Train MAPE', 'Test MAPE', 'Train R²', 'Test R²', 'CV RMSE', 'Time'], cellLoc='center', loc='center')
        table.auto_set_font_size(False); table.set_fontsize(10); table.scale(1.2, 2)
        for i in range(len(summary_data[0])): table[(0, i)].set_facecolor('#4ECDC4'); table[(0, i)].set_text_props(weight='bold', color='white')
        ax.axis('off')
        
        plt.tight_layout()
        plt.savefig(self.output_dir / f'performance_summary_table_{run_id}.png', bbox_inches='tight', facecolor='white')
        plt.close(fig)
        
        summary_file = self.output_dir / f'analysis_summary_{run_id}.txt'
        with open(summary_file, 'w') as f:
            f.write(f"ANALYSIS SUMMARY for min_records_per_well = {min_records_val}\n" + "="*50 + "\n")
            for name, res in results.items():
                f.write(f"\n{name}:\n" + "-"*15 + "\n")
                f.write(f"  Test MAPE: {res['test_metrics']['mape']:.2f}%\n")
                f.write(f"  Test R²: {res['test_metrics']['r2']:.3f}\n")
                f.write(f"  Training Time: {res['training_time_minutes']:.2f} minutes\n")

# Usage
def main():
    # Update this path to your actual file location
    file_path = r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\results\chile_2025\groundwater_chile_and_elevation_dataset_2025_with_GEE_7_2_25.xlsx"
    
    predictor = GroundwaterPredictor(file_path)
    predictor.load_and_preprocess_data()
    
    # Define the values for the sensitivity analysis
    min_records_values = [3, 9, 27, 54, 81]
    
    # Run the new sensitivity analysis workflow
    predictor.run_sensitivity_analysis(min_records_values)

if __name__ == "__main__":
    main()

## Key Changes Made:

### 1. **Focus on Optimal Configuration**
- Fixed `min_records_per_well = 9` (the best value from previous analysis)
- Dedicated hyperparameter optimization workflow

### 2. **Comprehensive Hyperparameter Grids**
- **Random Forest & Extra Trees**: `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`, `max_features`
- **Decision Tree**: `max_depth`, `min_samples_split`, `min_samples_leaf`, `criterion`, `max_features`

### 3. **Four-Step Optimization Process**
1. **Default Training**: Baseline performance with original hyperparameters
2. **Hyperparameter Search**: GridSearchCV with 5-fold cross-validation
3. **Optimized Training**: Performance with best hyperparameters
4. **Comparison & Visualization**: Side-by-side analysis

### 4. **Enhanced Visualizations**
- **Performance Comparison**: Bar charts showing default vs optimized metrics
- **Detailed 1-to-1 Plots**: Separate plots for default and optimized models
- **Impact Analysis**: Shows improvement metrics and training time changes
- **Comprehensive Report**: Text file with detailed optimization results

### 5. **Key Features**
- **Real Hyperparameter Values**: Shows actual optimized parameters
- **Performance Improvements**: Quantifies R², MAPE, and RMSE improvements
- **Training Time Impact**: Tracks computational cost changes
- **Visual Indicators**: Green highlighting for improvements in plots

### 6. **Expected Outputs**
- `performance_comparison.png`: Bar charts comparing metrics
- `detailed_comparison_plots.png`: 2×3 grid of 1-to-1 plots
- `hyperparameter_impact_analysis.png`: Improvement analysis
- `hyperparameter_optimization_report.txt`: Detailed text report

This approach will help you determine if hyperparameter optimization can significantly improve model performance beyond the optimal data splitting strategy you've already identified.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance
import warnings
import time
from pathlib import Path
import os
from datetime import datetime
warnings.filterwarnings('ignore')

# Set professional plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 9

class GroundwaterPredictor:
    """Professional groundwater depth prediction analysis with hyperparameter optimization."""
    
    def __init__(self, file_path):
        self.file_path = file_path
        self.df = None
        self.df_filtered = None
        self.results = {}
        self.predictions = {}
        self.feature_cols = []
        self.target_col = 'Depth to water (m)'
        
        # Default models
        self.default_models = {
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
            'Decision Tree': DecisionTreeRegressor(random_state=42)
        }
        
        # Hyperparameter grids for optimization
        self.param_grids = {
            'Extra Trees': {
                'n_estimators': [50, 100, 200],
                'max_depth': [None, 10, 20, 30],
                'min_samples_split': [2, 5, 10],
                'min_samples_leaf': [1, 2, 4],
                'max_features': ['sqrt', 'log2', None]
            },
            'Random Forest': {
                'n_estimators': [50, 100, 200],
                'max_depth': [None, 10, 20, 30],
                'min_samples_split': [2, 5, 10],
                'min_samples_leaf': [1, 2, 4],
                'max_features': ['sqrt', 'log2', None]
            },
            'Decision Tree': {
                'max_depth': [None, 5, 10, 15, 20],
                'min_samples_split': [2, 5, 10, 20],
                'min_samples_leaf': [1, 2, 4, 8],
                'criterion': ['squared_error', 'friedman_mse'],
                'max_features': ['sqrt', 'log2', None]
            }
        }
        
        # Timing variables
        self.start_time = None
        self.timing_log = []
        
        # Create output directory for figures
        self.output_dir = Path("groundwater_hyperparameter_optimization_results")
        self.output_dir.mkdir(exist_ok=True)
        
        # Store optimization results
        self.optimization_results = {}
        
    def log_time(self, step_name):
        """Log timing information for each step."""
        current_time = time.time()
        if self.start_time is None:
            self.start_time = current_time
            elapsed = 0
        else:
            elapsed = current_time - self.start_time
        
        self.timing_log.append({
            'step': step_name,
            'elapsed_minutes': elapsed / 60,
            'timestamp': datetime.now().strftime('%H:%M:%S')
        })
        
        print(f"⏱️  {step_name} completed at {datetime.now().strftime('%H:%M:%S')} "
              f"(Total elapsed: {elapsed/60:.2f} minutes)")
        
    def print_timing_summary(self):
        """Print a summary of timing information."""
        if not self.timing_log:
            return
            
        print("\n" + "="*60)
        print("⏰ TIMING SUMMARY")
        print("="*60)
        
        for i, entry in enumerate(self.timing_log):
            if i == 0:
                step_time = entry['elapsed_minutes']
            else:
                step_time = entry['elapsed_minutes'] - self.timing_log[i-1]['elapsed_minutes']
            
            print(f"{entry['step']:<40} | {step_time:>8.2f} min | {entry['timestamp']}")
        
        total_time = self.timing_log[-1]['elapsed_minutes']
        print("-" * 60)
        print(f"{'TOTAL RUNTIME':<40} | {total_time:>8.2f} min")
        print("="*60)
    
    def mean_absolute_percentage_error(self, y_true, y_pred):
        """Calculate Mean Absolute Percentage Error (MAPE)"""
        epsilon = 1e-10
        return np.mean(np.abs((y_true - y_pred) / (y_true + epsilon))) * 100
    
    def load_and_preprocess_data(self):
        """Load and preprocess the dataset."""
        print("🌊 Chile Groundwater Depth Prediction Analysis")
        print("🔧 Focus: Hyperparameter Optimization (min_records_per_well = 9)")
        print("=" * 70)
        print(f"🚀 Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("\n📊 Loading and preprocessing dataset...")
        
        self.log_time("Analysis Started")
        
        try:
            if self.file_path.endswith('.csv'):
                self.df = pd.read_csv(self.file_path)
            else:
                self.df = pd.read_excel(self.file_path)
            print(f"✅ Dataset loaded successfully! Shape: {self.df.shape}")
        except Exception as e:
            print(f"❌ Error loading file: {e}")
            # Create sample data for demonstration
            print("Creating sample data for demonstration...")
            np.random.seed(42)
            n_samples = 10000
            n_wells = 1000
            records_per_well = np.random.poisson(15, n_wells) + 1
            well_data = []
            for i in range(n_wells):
                well_code = f'Well_{i:04d}'
                n_records = records_per_well[i]
                base_lon = np.random.uniform(-75, -65); base_lat = np.random.uniform(-35, -25); base_elevation = np.random.normal(500, 200)
                for j in range(n_records):
                    date_value = pd.Timestamp('2010-01-01') + pd.Timedelta(days=j*90 + np.random.randint(0, 90))
                    well_data.append({
                        'Code': well_code, 'Date_String': date_value, 'Depth to water (m)': np.random.exponential(10) + j*0.1,
                        'Longitude_GCS_WGS_1984': base_lon + np.random.normal(0, 0.01), 'Latitude_GCS_WGS_1984': base_lat + np.random.normal(0, 0.01),
                        'elevation_NASADEM': base_elevation + np.random.normal(0, 50), 'slope_NASADEM': np.random.exponential(5),
                        'terraclim_pr_value': np.random.exponential(100), 'terraclim_tmmn_value': np.random.normal(100, 50),
                        'terraclim_tmmx_value': np.random.normal(200, 50), 'Basin': np.random.choice(['Basin_A', 'Basin_B', 'Basin_C']),
                        'Status': np.random.choice(['Active', 'Inactive', np.nan], p=[0.3, 0.3, 0.4])
                    })
            self.df = pd.DataFrame(well_data)

        self.log_time("Data Loading")
        
        print(f"Original dataset shape: {self.df.shape}")
        self.df_filtered = self.df[self.df['Status'].isnull()].copy()
        print(f"Filtered dataset shape (Status is blank): {self.df_filtered.shape}")
        
        if len(self.df_filtered) == 0:
            print("❌ No rows with blank Status found! Using all data for demonstration.")
            self.df_filtered = self.df.copy()
        
        initial_rows = len(self.df_filtered)
        self.df_filtered = self.df_filtered.dropna(subset=[self.target_col])
        print(f"Removed {initial_rows - len(self.df_filtered)} rows with missing target values")
        
        self._prepare_features()
        self.log_time("Data Preprocessing")
        
    def _prepare_features(self):
        """Prepare feature matrix and target variable."""
        numeric_cols = self.df_filtered.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = self.df_filtered.select_dtypes(include=['object']).columns.tolist()
        
        if self.target_col in numeric_cols:
            numeric_cols.remove(self.target_col)
        
        columns_to_remove = ['Status', 'Code', 'Date_String']
        for col in columns_to_remove:
            if col in categorical_cols: categorical_cols.remove(col)
            if col in numeric_cols: numeric_cols.remove(col)
        
        for col in numeric_cols:
            if self.df_filtered[col].isnull().sum() > 0: self.df_filtered[col].fillna(self.df_filtered[col].median(), inplace=True)
        
        for col in categorical_cols:
            if col in self.df_filtered.columns:
                if self.df_filtered[col].isnull().sum() > 0: self.df_filtered[col].fillna(self.df_filtered[col].mode()[0], inplace=True)
                le = LabelEncoder()
                self.df_filtered[col + '_encoded'] = le.fit_transform(self.df_filtered[col].astype(str))
        
        encoded_categorical_cols = [col + '_encoded' for col in categorical_cols if col in self.df_filtered.columns]
        self.feature_cols = numeric_cols + encoded_categorical_cols
        print(f"Final feature set: {len(self.feature_cols)} features")
        
    def create_improved_temporal_well_split(self, X, y, train_ratio=0.7, min_records_per_well=9):
        """Improved temporal split ensuring minimum records per well."""
        if 'Code' not in self.df_filtered.columns:
            print("Warning: 'Code' column not found. Using random split instead.")
            return train_test_split(X, y, test_size=0.3, random_state=42)
        
        if 'Date_String' not in self.df_filtered.columns:
            print("Creating synthetic date ordering based on data sequence...")
            self.df_filtered['Date_String'] = self.df_filtered.groupby('Code').cumcount()
        
        train_indices, test_indices, wells_with_insufficient_data = [], [], []
        unique_wells = self.df_filtered['Code'].unique()
        
        for well in unique_wells:
            well_mask = self.df_filtered['Code'] == well
            well_data_sorted = self.df_filtered[well_mask].copy().sort_values('Date_String')
            well_indices = well_data_sorted.index.tolist()
            n_records = len(well_indices)
            
            if n_records < min_records_per_well:
                wells_with_insufficient_data.append(well)
                train_indices.extend(well_indices)
                continue
            
            n_train = max(1, int(n_records * train_ratio))
            n_train = min(n_train, n_records - 1)
            train_indices.extend(well_indices[:n_train])
            test_indices.extend(well_indices[n_train:])
        
        print(f"Wells with insufficient data (< {min_records_per_well} records): {len(wells_with_insufficient_data)} (used for training)")
        print(f"Training records: {len(train_indices)}")
        print(f"Testing records: {len(test_indices)}")
        
        train_mask = self.df_filtered.index.isin(train_indices)
        test_mask = self.df_filtered.index.isin(test_indices)
        return X[train_mask], X[test_mask], y[train_mask], y[test_mask]
        
    def run_hyperparameter_optimization(self):
        """Run complete hyperparameter optimization analysis."""
        X = self.df_filtered[self.feature_cols].values
        y = self.df_filtered[self.target_col].values
        
        print(f"\n{'='*70}")
        print(f"🔧 HYPERPARAMETER OPTIMIZATION ANALYSIS")
        print(f"🔧 Using optimal min_records_per_well = 9")
        print(f"{'='*70}")
        
        # Create train/test split using optimal parameters
        X_train, X_test, y_train, y_test = self.create_improved_temporal_well_split(
            X, y, train_ratio=0.7, min_records_per_well=9
        )
        
        print(f"Training samples: {len(X_train)} ({len(X_train)/(len(X_train)+len(X_test))*100:.1f}%)")
        print(f"Testing samples: {len(X_test)} ({len(X_test)/(len(X_train)+len(X_test))*100:.1f}%)")
        
        # Step 1: Train with default hyperparameters
        print("\n" + "="*50)
        print("🔧 STEP 1: Training with DEFAULT hyperparameters")
        print("="*50)
        default_results = self._train_and_evaluate_models(X_train, X_test, y_train, y_test, self.default_models, "Default")
        
        # Step 2: Optimize hyperparameters
        print("\n" + "="*50)
        print("🔧 STEP 2: HYPERPARAMETER OPTIMIZATION")
        print("="*50)
        optimized_models = self._optimize_hyperparameters(X_train, y_train)
        
        # Step 3: Train with optimized hyperparameters
        print("\n" + "="*50)
        print("🔧 STEP 3: Training with OPTIMIZED hyperparameters")
        print("="*50)
        optimized_results = self._train_and_evaluate_models(X_train, X_test, y_train, y_test, optimized_models, "Optimized")
        
        # Step 4: Compare results
        print("\n" + "="*50)
        print("🔧 STEP 4: COMPARISON AND VISUALIZATION")
        print("="*50)
        self._compare_and_visualize_results(default_results, optimized_results, y_train, y_test)
        
        self.log_time("Hyperparameter Optimization Completed")
        self.print_timing_summary()
        print(f"\n🎉 Hyperparameter optimization completed! All results saved in '{self.output_dir}' directory")
        
    def _optimize_hyperparameters(self, X_train, y_train):
        """Optimize hyperparameters using GridSearchCV."""
        optimized_models = {}
        
        for model_name in self.default_models.keys():
            print(f"\n🔍 Optimizing {model_name}...")
            start_time = time.time()
            
            # Get base model
            if model_name == 'Extra Trees':
                base_model = ExtraTreesRegressor(random_state=42, n_jobs=-1)
            elif model_name == 'Random Forest':
                base_model = RandomForestRegressor(random_state=42, n_jobs=-1)
            elif model_name == 'Decision Tree':
                base_model = DecisionTreeRegressor(random_state=42)
            
            # Perform grid search
            grid_search = GridSearchCV(
                estimator=base_model,
                param_grid=self.param_grids[model_name],
                cv=5,
                scoring='neg_mean_squared_error',
                n_jobs=-1,
                verbose=1
            )
            
            grid_search.fit(X_train, y_train)
            
            # Store results
            optimized_models[model_name] = grid_search.best_estimator_
            optimization_time = (time.time() - start_time) / 60
            
            print(f"  ✅ {model_name} optimization completed in {optimization_time:.2f} minutes")
            print(f"  🎯 Best CV Score (RMSE): {np.sqrt(-grid_search.best_score_):.3f}")
            print(f"  ⚙️  Best Parameters: {grid_search.best_params_}")
            
            # Store optimization details
            self.optimization_results[model_name] = {
                'best_params': grid_search.best_params_,
                'best_score': grid_search.best_score_,
                'optimization_time': optimization_time
            }
        
        return optimized_models
    
    def _train_and_evaluate_models(self, X_train, X_test, y_train, y_test, models, prefix):
        """Train and evaluate all models."""
        results = {}
        
        for name, model in models.items():
            model_start_time = time.time()
            print(f"Training {prefix} {name}...")
            
            model.fit(X_train, y_train)
            y_pred_train, y_pred_test = model.predict(X_train), model.predict(X_test)
            train_metrics, test_metrics = self._calculate_metrics(y_train, y_pred_train), self._calculate_metrics(y_test, y_pred_test)
            cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
            model_time = (time.time() - model_start_time) / 60
            
            print(f"  ✅ {prefix} {name} completed in {model_time:.2f} minutes")
            print(f"  📊 Test R²: {test_metrics['r2']:.3f}, Test MAPE: {test_metrics['mape']:.2f}%")
            
            results[name] = {
                'model': model, 'train_metrics': train_metrics, 'test_metrics': test_metrics,
                'cv_rmse': np.sqrt(-cv_scores.mean()), 'train_predictions': y_pred_train, 
                'test_predictions': y_pred_test, 'training_time_minutes': model_time
            }
        
        return results
    
    def _calculate_metrics(self, y_true, y_pred):
        """Calculate performance metrics."""
        return {
            'rmse': np.sqrt(mean_squared_error(y_true, y_pred)), 'mae': mean_absolute_error(y_true, y_pred),
            'r2': r2_score(y_true, y_pred), 'mape': self.mean_absolute_percentage_error(y_true, y_pred)
        }
    
    def _compare_and_visualize_results(self, default_results, optimized_results, y_train, y_test):
        """Compare and visualize default vs optimized results."""
        # Create comparison summary
        self._create_comparison_summary(default_results, optimized_results)
        
        # Create detailed performance comparison plots
        self._create_performance_comparison_plots(default_results, optimized_results)
        
        # Create 1-to-1 comparison plots
        self._create_detailed_comparison_plots(default_results, optimized_results, y_train, y_test)
        
        # Create hyperparameter impact analysis
        self._create_hyperparameter_impact_analysis(default_results, optimized_results)
        
        # Save detailed comparison report
        self._save_optimization_report(default_results, optimized_results)
    
    def _create_comparison_summary(self, default_results, optimized_results):
        """Create a summary table comparing default vs optimized performance."""
        print("\n📈 PERFORMANCE COMPARISON SUMMARY")
        print("="*80)
        
        summary_data = []
        for model_name in default_results.keys():
            default_metrics = default_results[model_name]['test_metrics']
            optimized_metrics = optimized_results[model_name]['test_metrics']
            
            r2_improvement = optimized_metrics['r2'] - default_metrics['r2']
            mape_improvement = default_metrics['mape'] - optimized_metrics['mape']  # Lower is better
            rmse_improvement = default_metrics['rmse'] - optimized_metrics['rmse']  # Lower is better
            
            summary_data.append([
                model_name,
                f"{default_metrics['r2']:.3f}",
                f"{optimized_metrics['r2']:.3f}",
                f"{r2_improvement:+.3f}",
                f"{default_metrics['mape']:.2f}%",
                f"{optimized_metrics['mape']:.2f}%",
                f"{mape_improvement:+.2f}%",
                f"{default_metrics['rmse']:.3f}",
                f"{optimized_metrics['rmse']:.3f}",
                f"{rmse_improvement:+.3f}"
            ])
        
        # Create DataFrame for better formatting
        columns = ['Model', 'Default R²', 'Optimized R²', 'R² Δ', 
                  'Default MAPE', 'Optimized MAPE', 'MAPE Δ',
                  'Default RMSE', 'Optimized RMSE', 'RMSE Δ']
        
        summary_df = pd.DataFrame(summary_data, columns=columns)
        print(summary_df.to_string(index=False))
        print("="*80)
    
    def _create_performance_comparison_plots(self, default_results, optimized_results):
        """Create bar charts comparing default vs optimized performance."""
        model_names = list(default_results.keys())
        metrics = ['r2', 'mape', 'rmse']
        metric_labels = ['R² Score', 'MAPE (%)', 'RMSE']
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        fig.suptitle('Performance Comparison: Default vs Optimized Hyperparameters', 
                     fontsize=16, fontweight='bold')
        
        x = np.arange(len(model_names))
        width = 0.35
        
        for i, (metric, label) in enumerate(zip(metrics, metric_labels)):
            default_scores = [default_results[name]['test_metrics'][metric] for name in model_names]
            optimized_scores = [optimized_results[name]['test_metrics'][metric] for name in model_names]
            
            bars1 = axes[i].bar(x - width/2, default_scores, width, label='Default', 
                               color='lightblue', alpha=0.8, edgecolor='navy')
            bars2 = axes[i].bar(x + width/2, optimized_scores, width, label='Optimized', 
                               color='lightcoral', alpha=0.8, edgecolor='darkred')
            
            axes[i].set_xlabel('Models')
            axes[i].set_ylabel(label)
            axes[i].set_title(f'{label} Comparison')
            axes[i].set_xticks(x)
            axes[i].set_xticklabels(model_names, rotation=45)
            axes[i].legend()
            axes[i].grid(True, alpha=0.3)
            
            # Add value labels on bars
            for bars in [bars1, bars2]:
                for bar in bars:
                    height = bar.get_height()
                    axes[i].text(bar.get_x() + bar.get_width()/2., height,
                               f'{height:.3f}' if metric != 'mape' else f'{height:.2f}%',
                               ha='center', va='bottom', fontsize=8)
        
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.savefig(self.output_dir / 'performance_comparison.png', bbox_inches='tight', facecolor='white')
        plt.close(fig)
    
    def _create_detailed_comparison_plots(self, default_results, optimized_results, y_train, y_test):
        """Create detailed 1-to-1 comparison plots."""
        model_names = list(default_results.keys())
        
        # Create a large figure comparing default vs optimized for each model
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        fig.suptitle('1-to-1 Plots: Default vs Optimized Hyperparameters', 
                     fontsize=16, fontweight='bold')
        
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
        
        for i, (model_name, color) in enumerate(zip(model_names, colors)):
            # Default model plot
            ax_default = axes[0, i]
            default_res = default_results[model_name]
            
            ax_default.scatter(y_train, default_res['train_predictions'], 
                             alpha=0.4, color='lightblue', s=15, 
                             edgecolors='navy', linewidth=0.3, 
                             label=f'Training (n={len(y_train):,})', marker='o')
            
            ax_default.scatter(y_test, default_res['test_predictions'], 
                             alpha=0.7, color=color, s=20, 
                             edgecolors='white', linewidth=0.5, 
                             label=f'Testing (n={len(y_test):,})', marker='s')
            
            all_values = np.concatenate([y_train, y_test])
            min_val, max_val = all_values.min(), all_values.max()
            ax_default.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='1:1 Line')
            
            ax_default.set_xlabel('Actual Depth to Water (m)')
            ax_default.set_ylabel('Predicted Depth to Water (m)')
            ax_default.set_title(f'{model_name} - Default Hyperparameters')
            ax_default.legend(loc='lower right', fontsize=8)
            ax_default.set_aspect('equal', adjustable='box')
            ax_default.grid(True, alpha=0.3)
            
            # Add metrics
            default_metrics = default_res['test_metrics']
            metrics_text = f'Test R² = {default_metrics["r2"]:.3f}\nTest MAPE = {default_metrics["mape"]:.2f}%'
            ax_default.text(0.05, 0.95, metrics_text, transform=ax_default.transAxes, 
                           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8), 
                           verticalalignment='top', fontsize=9)
            
            # Optimized model plot
            ax_optimized = axes[1, i]
            optimized_res = optimized_results[model_name]
            
            ax_optimized.scatter(y_train, optimized_res['train_predictions'], 
                               alpha=0.4, color='lightblue', s=15, 
                               edgecolors='navy', linewidth=0.3, 
                               label=f'Training (n={len(y_train):,})', marker='o')
            
            ax_optimized.scatter(y_test, optimized_res['test_predictions'], 
                               alpha=0.7, color=color, s=20, 
                               edgecolors='white', linewidth=0.5, 
                               label=f'Testing (n={len(y_test):,})', marker='s')
            
            ax_optimized.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='1:1 Line')
            
            ax_optimized.set_xlabel('Actual Depth to Water (m)')
            ax_optimized.set_ylabel('Predicted Depth to Water (m)')
            ax_optimized.set_title(f'{model_name} - Optimized Hyperparameters')
            ax_optimized.legend(loc='lower right', fontsize=8)
            ax_optimized.set_aspect('equal', adjustable='box')
            ax_optimized.grid(True, alpha=0.3)
            
            # Add metrics and improvement
            optimized_metrics = optimized_res['test_metrics']
            r2_improvement = optimized_metrics['r2'] - default_metrics['r2']
            mape_improvement = default_metrics['mape'] - optimized_metrics['mape']
            
            metrics_text = (f'Test R² = {optimized_metrics["r2"]:.3f} ({r2_improvement:+.3f})\n'
                          f'Test MAPE = {optimized_metrics["mape"]:.2f}% ({mape_improvement:+.2f}%)')
            ax_optimized.text(0.05, 0.95, metrics_text, transform=ax_optimized.transAxes, 
                            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8), 
                            verticalalignment='top', fontsize=9)
        
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.savefig(self.output_dir / 'detailed_comparison_plots.png', bbox_inches='tight', facecolor='white')
        plt.close(fig)
    
    def _create_hyperparameter_impact_analysis(self, default_results, optimized_results):
        """Create visualization showing hyperparameter impact."""
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Hyperparameter Optimization Impact Analysis', fontsize=16, fontweight='bold')
        
        model_names = list(default_results.keys())
        
        # Performance improvement metrics
        improvements = {
            'R² Improvement': [],
            'MAPE Improvement (%)': [],
            'RMSE Improvement': [],
            'Training Time Change (min)': []
        }
        
        for model_name in model_names:
            default_metrics = default_results[model_name]['test_metrics']
            optimized_metrics = optimized_results[model_name]['test_metrics']
            
            improvements['R² Improvement'].append(optimized_metrics['r2'] - default_metrics['r2'])
            improvements['MAPE Improvement (%)'].append(default_metrics['mape'] - optimized_metrics['mape'])
            improvements['RMSE Improvement'].append(default_metrics['rmse'] - optimized_metrics['rmse'])
            improvements['Training Time Change (min)'].append(
                optimized_results[model_name]['training_time_minutes'] - 
                default_results[model_name]['training_time_minutes']
            )
        
        # Plot improvements
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
        
        for i, (metric, values) in enumerate(improvements.items()):
            if i < 4:  # We have 4 subplots
                row, col = i // 2, i % 2
                ax = axes[row, col]
                
                bars = ax.bar(model_names, values, color=colors, alpha=0.8)
                ax.set_title(metric)
                ax.set_ylabel('Improvement')
                ax.tick_params(axis='x', rotation=45)
                ax.grid(True, alpha=0.3)
                ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
                
                # Add value labels
                for bar, value in zip(bars, values):
                    height = bar.get_height()
                    ax.text(bar.get_x() + bar.get_width()/2., height,
                           f'{value:+.3f}' if 'Time' not in metric else f'{value:+.2f}',
                           ha='center', va='bottom' if height >= 0 else 'top', 
                           fontweight='bold')
        
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        plt.savefig(self.output_dir / 'hyperparameter_impact_analysis.png', bbox_inches='tight', facecolor='white')
        plt.close(fig)
    
    def _save_optimization_report(self, default_results, optimized_results):
        """Save detailed optimization report to text file."""
        report_file = self.output_dir / 'hyperparameter_optimization_report.txt'
        
        with open(report_file, 'w') as f:
            f.write("HYPERPARAMETER OPTIMIZATION REPORT\n")
            f.write("="*60 + "\n")
            f.write(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"Optimal min_records_per_well: 9\n\n")
            
            for model_name in default_results.keys():
                f.write(f"\n{model_name}\n")
                f.write("-" * len(model_name) + "\n")
                
                # Default performance
                default_metrics = default_results[model_name]['test_metrics']
                f.write(f"DEFAULT HYPERPARAMETERS:\n")
                f.write(f"  Test R²: {default_metrics['r2']:.3f}\n")
                f.write(f"  Test MAPE: {default_metrics['mape']:.2f}%\n")
                f.write(f"  Test RMSE: {default_metrics['rmse']:.3f}\n")
                f.write(f"  Training Time: {default_results[model_name]['training_time_minutes']:.2f} min\n\n")
                
                # Optimized performance
                optimized_metrics = optimized_results[model_name]['test_metrics']
                f.write(f"OPTIMIZED HYPERPARAMETERS:\n")
                if model_name in self.optimization_results:
                    f.write(f"  Best Parameters: {self.optimization_results[model_name]['best_params']}\n")
                f.write(f"  Test R²: {optimized_metrics['r2']:.3f}\n")
                f.write(f"  Test MAPE: {optimized_metrics['mape']:.2f}%\n")
                f.write(f"  Test RMSE: {optimized_metrics['rmse']:.3f}\n")
                f.write(f"  Training Time: {optimized_results[model_name]['training_time_minutes']:.2f} min\n\n")
                
                # Improvements
                r2_improvement = optimized_metrics['r2'] - default_metrics['r2']
                mape_improvement = default_metrics['mape'] - optimized_metrics['mape']
                rmse_improvement = default_metrics['rmse'] - optimized_metrics['rmse']
                
                f.write(f"IMPROVEMENTS:\n")
                f.write(f"  R² Improvement: {r2_improvement:+.3f}\n")
                f.write(f"  MAPE Improvement: {mape_improvement:+.2f}%\n")
                f.write(f"  RMSE Improvement: {rmse_improvement:+.3f}\n\n")

# Usage
def main():
    # Update this path to your actual file location
    file_path = r"E:\OneDrive - Colorado School of Mines\projects_2025\chile_groundwater\results\chile_2025\groundwater_chile_and_elevation_dataset_2025_with_GEE_7_2_25.xlsx"
    
    predictor = GroundwaterPredictor(file_path)
    predictor.load_and_preprocess_data()
    
    # Run hyperparameter optimization using the optimal min_records_per_well = 9
    predictor.run_hyperparameter_optimization()

if __name__ == "__main__":
    main()